# 01 · 方法

全项目共用的函数都在这里：数据库读取、Copula、VaR/ES、配置、模拟、预警、拥挤度。

**用法**
- 各研究 notebook 的第一格用 `%run ../../01_methods.ipynb` 引入这里的全部函数。
- 直接打开本 notebook 从头运行，末尾的「自检」会跑一遍回归断言（约 1 分钟）；被 `%run` 引入时自动跳过自检。
- 数据先由 [00_data.ipynb](00_data.ipynb) 取到本地 SQLite（`data/tailrisk.db`），这里只读不写行情。

In [1]:
from __future__ import annotations

import sqlite3
from contextlib import closing
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.optimize import curve_fit, linprog, minimize, minimize_scalar
from scipy.spatial.distance import squareform
from scipy.special import gammaln
from scipy.stats import chi2, fisher_exact, genpareto, kendalltau, norm, rankdata, t as student_t

# 被研究 notebook 用 %run 引入时，IPython 会设置 __file__；直接打开本 notebook 时没有
RUN_SELFTEST = "__file__" not in globals()

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "01_methods.ipynb").exists())
DB_PATH = ROOT / "data" / "tailrisk.db"

## 0. 数据库读取

行情由 `00_data.ipynb` 写入 `data/tailrisk.db`：
- `prices(name, date, value)`：日频价格 / 指数值 / 利率，一行一个（序列, 日期）
- `french_factors(date, factor, value)`：Kenneth French 月度因子收益（小数）
- `series_info`：每条序列的来源与更新时间

In [2]:
# 东北证券（2025）示例中的 10 只 ETF
ETF10 = ["沪深300ETF", "中证500ETF", "中证1000ETF", "创业板ETF", "消费ETF",
         "黄金ETF", "纳指ETF", "恒生ETF", "红利ETF", "大宗商品ETF"]
FRENCH_COLS = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom", "ST_Rev", "LT_Rev"]


def _db():
    if not DB_PATH.exists():
        raise FileNotFoundError(f"找不到 {DB_PATH}：先运行项目根目录的 00_data.ipynb")
    return closing(sqlite3.connect(DB_PATH))


def _query(sql, params=()):
    with _db() as con:
        return pd.read_sql(sql, con, params=params)


def load_series(name):
    """按名称读一条日频序列。"""
    df = _query("SELECT date, value FROM prices WHERE name = ? ORDER BY date", (name,))
    if df.empty:
        raise KeyError(f"数据库里没有「{name}」：在 00_data.ipynb 的 SERIES 里登记后重新运行")
    return pd.Series(df["value"].values, index=pd.DatetimeIndex(df["date"].values), name=name)


def load_prices(names, start=None, end=None, how="inner"):
    """
    取多条序列并按日期对齐。默认 inner：只保留共同交易日。

    不做前向填充 —— 填充会制造零收益（ties），让 Copula 的 MLE 偏向独立解。
    """
    df = pd.concat([load_series(n) for n in names], axis=1, join=how)
    return df.loc[start:end].sort_index()


def load_sw_industries():
    """申万一级行业收盘点位，一列一个行业，列名为行业代码。"""
    df = _query(r"SELECT name, date, value FROM prices WHERE name LIKE '申万\_%' ESCAPE '\'")
    df["code"] = df["name"].str[3:]
    out = df.pivot(index="date", columns="code", values="value")
    out.index = pd.to_datetime(out.index)
    out.columns.name = None
    return out.sort_index()


def load_french_factors():
    """八个美股因子的月度收益（小数），index 为月末。"""
    df = _query("SELECT date, factor, value FROM french_factors")
    out = df.pivot(index="date", columns="factor", values="value")[FRENCH_COLS]
    out.index = pd.to_datetime(out.index)
    out.index.name = out.columns.name = None
    return out.sort_index()


def to_returns(prices):
    """
    简单收益率。显式 fill_method=None：pandas 的 pct_change 默认会对 NaN
    做前向填充，缺失日变成 0 收益、下一日跨缺口计收益，恰好制造 ties。
    """
    return prices.pct_change(fill_method=None).dropna(how="all")


In [3]:
def data_quality(df):
    """
    数据质量体检。返回 DataFrame，一行一个标的。

    关注 ties（零收益率、重复值）：PIT 建立在经验分布上，大量重复值会让
    多个观测映射到同一分位，Copula 的似然曲面趋于平坦，MLE 系统性偏向
    独立解 —— 表现为 lambda 恒为 0，与「真的没有尾部相关」无法区分。
    """
    rows = []
    for c in df.columns:
        s = df[c].dropna()
        if len(s) < 3:
            rows.append({"标的": c, "样本数": len(s), "可用": False})
            continue
        r = s.pct_change().dropna()
        n_zero = int((r.abs() < 1e-10).sum())
        rows.append({
            "标的": c,
            "样本数": len(s),
            "起点": s.index.min().date(),
            "终点": s.index.max().date(),
            "年化收益": (s.iloc[-1] / s.iloc[0]) ** (365.25 / (s.index[-1] - s.index[0]).days) - 1,
            "年化波动": r.std() * np.sqrt(244),
            "最大回撤": (s / s.cummax() - 1).min(),
            "零收益占比": n_zero / len(r),
            "唯一值占比": r.nunique() / len(r),
            "可用": (n_zero / len(r) < 0.05) and (r.nunique() / len(r) > 0.80),
        })
    return pd.DataFrame(rows)

## 1. 边缘分布与 PIT

```text
边缘分布与概率积分变换（PIT）。

中金原文没有说明边缘分布怎么处理，只说尾部相关系数「使用过去 60 个交易日
数据计算」。本项目据此把**窗口内秩变换**（rank_pit，在 copula.rolling_tail_dependence
里逐窗口调用）作为主口径：只用窗口内数据，天然没有前视偏差。

下面三个全样本函数作为对照口径保留：
    pit_empirical   全样本秩变换 —— 用到了未来数据，有前视偏差
    pit_garch       全样本 GJR-GARCH 滤波后秩变换 —— 同上，且会剥离波动率联动
    pit_vol_scaled  滚动波动率标准化后全样本秩变换
```

In [4]:
def rank_pit(x):
    """
    秩变换到开区间 (0,1)：rank/(n+1)，ties 取平均秩。
    除以 n+1 而非 n，避免 Copula 对数似然出现 log(0)。
    """
    return rankdata(x, method="average") / (len(x) + 1)


def pit_empirical(r):
    """全样本经验 CDF 变换。⚠ 每个点的分位都依赖全样本，含未来信息。"""
    s = pd.Series(r).dropna()
    return pd.Series(rank_pit(s.values), index=s.index)


def pit_garch(r, dist="skewt"):
    """
    GJR-GARCH 滤波 + 标准化残差的经验 PIT。⚠ 全样本拟合，含未来信息。

    危机期「股债同步暴跌」在数据上很大程度表现为两者波动率同时放大，
    GARCH 滤波恰好剥离这一层，可能把要测的东西一起剥掉。只作对照。
    """
    from arch import arch_model

    s = pd.Series(r).dropna()
    am = arch_model(s * 100, vol="GARCH", p=1, o=1, q=1, dist=dist)
    res = am.fit(disp="off", show_warning=False)
    z = res.std_resid.dropna()
    return pd.Series(rank_pit(z.values), index=z.index)


def pit_vol_scaled(r, window=60):
    """
    滚动波动率标准化后再做经验 PIT。介于上面两者之间的中间档。

    波动率用 shift(1)：当日收益不参与自己的标准化。
    但最后一步秩变换仍是全样本的。
    """
    s = pd.Series(r).dropna()
    vol = s.rolling(window, min_periods=window // 2).std().shift(1)
    z = (s / vol).dropna()
    return pd.Series(rank_pit(z.values), index=z.index)


def es_hist(r, q=0.95):
    """历史法 ES：损失（−收益）中超过 q 分位 VaR 的部分的均值。返回正数（损失）。"""
    loss = -np.asarray(pd.Series(r).dropna(), dtype=float)
    var = np.quantile(loss, q)
    return float(loss[loss >= var].mean())


def es_gpd(r, q=0.95, threshold=0.90):
    """
    POT-GPD 法 ES：对超过 threshold 分位的损失超额拟合广义帕累托分布（位置固定为 0），
        VaR_q = u + β/ξ·[((n/N_u)(1−q))^(−ξ) − 1]
        ES_q  = (VaR_q + β − ξu) / (1 − ξ)          （ξ < 1）
    ξ≈0 时取指数分布极限：VaR_q = u − β·ln((n/N_u)(1−q))，ES_q = VaR_q + β。
    """
    from scipy.stats import genpareto

    loss = -np.asarray(pd.Series(r).dropna(), dtype=float)
    n = len(loss)
    u = np.quantile(loss, threshold)
    exc = loss[loss > u] - u
    xi, _, beta = genpareto.fit(exc, floc=0)
    ratio = n / len(exc) * (1 - q)
    if abs(xi) < 1e-6:
        var = u - beta * np.log(ratio)
        return float(var + beta)
    var = u + beta / xi * (ratio ** (-xi) - 1)
    return float((var + beta - xi * u) / (1 - xi)) if xi < 1 else np.inf

## 2. Copula 与滚动尾部相关

```text
Copula 估计、滚动尾部相关系数、噪声基准。

══ 「λ=0」的两种判定口径 ═══════════════════════════════════════════

中金原文没有给出任何检验，只写了 Clayton 参数→0、Gumbel 参数→1 时两变量不相关。
因此「λ=0」最贴近原文的理解是**点估计落在独立边界上**。实现上，λ 是 θ 的连续
函数，θ 稍离边界时 λ 就小到可以忽略（Clayton θ=0.05 时 λ_D=2^-20≈1e-6），
所以用一个显示精度阈值 LAMBDA_ZERO_TOL 判零：

    "point"   λ < LAMBDA_ZERO_TOL 即为 0（原文口径，主口径）
    "lrtest"  另外要求独立性似然比检验显著，否则判 0（本项目加的保守口径）

══ 已知的方法局限（两种口径都有）═════════════════════════════════

Clayton/Gumbel 都只是「有正相依」的单参数模型。只要窗口内整体正相关，
MLE 就会给出离开边界的 θ，从而 λ>0 —— 即使真实的尾部相关严格为 0。
用 gaussian_null 做对照可以看到：高斯 copula（无尾部相关）在 ρ=0.3 时，
λ_D>0 的窗口占一半以上。所以 λ_D>0 的频率必须和
「相同相关强度、无尾部相关」的基准对比（expected_under_gaussian），
不能只和独立数据比。
```

In [5]:
_EPS = 1e-6


LAMBDA_ZERO_TOL = 1e-4      # 万分之一：原文 λ 以百分比两位小数展示，低于此即显示为 0.00%


ZERO_RULES = ("point", "lrtest")


def _clip01(x):
    return np.clip(np.asarray(x, dtype=float), _EPS, 1 - _EPS)


def _lr_pvalue(loglik):
    """
    独立性似然比检验 p 值。独立时两族的对数似然都恰为 0。

    θ 的独立值在参数空间边界上，严格的零分布是 χ²(0) 与 χ²(1) 的 50:50 混合；
    用纯 χ²(1) 偏保守（独立数据的实测误报率约 2%，低于名义 5%）。
    """
    return float(chi2.sf(2 * max(loglik, 0.0), df=1))


class ClaytonCopula:
    """仅下尾相关：λ_D = 2^(-1/θ)，θ→0 对应独立。"""

    name = "clayton"
    bounds = (1e-6, 30.0)

    @staticmethod
    def loglik_terms(theta, u, v):
        lu, lv = np.log(u), np.log(v)
        # log(u^-θ + v^-θ - 1) 在 log 空间计算，θ 大时 u^-θ 会溢出
        s = np.logaddexp(-theta * lu, -theta * lv)
        log_inner = s + np.log1p(-np.exp(-s))
        return np.log1p(theta) - (theta + 1) * (lu + lv) - (1 / theta + 2) * log_inner

    def fit(self, u, v):
        u, v = _clip01(u), _clip01(v)
        res = minimize_scalar(lambda t: -np.sum(self.loglik_terms(t, u, v)),
                              bounds=self.bounds, method="bounded")
        self.theta = float(res.x)
        self.loglik = float(-res.fun)
        self.pvalue = _lr_pvalue(self.loglik)
        self.lambda_raw = float(2 ** (-1 / self.theta))
        return self


class GumbelCopula:
    """仅上尾相关：λ_U = 2 - 2^(1/θ)，θ→1 对应独立。"""

    name = "gumbel"
    bounds = (1.0, 30.0)

    @staticmethod
    def loglik_terms(theta, u, v):
        x, y = -np.log(u), -np.log(v)
        lx, ly = np.log(x), np.log(y)
        log_A = np.logaddexp(theta * lx, theta * ly)
        A_inv = np.exp(log_A / theta)
        return (-A_inv + x + y + (theta - 1) * (lx + ly)
                + (1 / theta - 2) * log_A + np.log(theta - 1 + A_inv))

    def fit(self, u, v):
        u, v = _clip01(u), _clip01(v)
        res = minimize_scalar(lambda t: -np.sum(self.loglik_terms(t, u, v)),
                              bounds=self.bounds, method="bounded")
        self.theta = float(res.x)
        self.loglik = float(-res.fun)
        self.pvalue = _lr_pvalue(self.loglik)
        self.lambda_raw = float(2 - 2 ** (1 / self.theta))
        return self

In [6]:
class GaussianCopula:
    """基准模型，尾部相关系数恒为 0 —— 这正是它低估尾部风险的原因。"""

    name = "gaussian"

    def fit(self, u, v):
        u, v = _clip01(u), _clip01(v)
        x, y = norm.ppf(u), norm.ppf(v)
        self.rho = float(np.corrcoef(x, y)[0, 1])
        det = 1 - self.rho ** 2
        self.loglik = float(np.sum(
            -0.5 * np.log(det)
            - (self.rho ** 2 * (x ** 2 + y ** 2) - 2 * self.rho * x * y) / (2 * det)))
        return self


def zero_out(lambda_raw, pvalue, rule="point", alpha=0.05, tol=LAMBDA_ZERO_TOL):
    """按判零口径把 λ 原始估计置零。标量或数组均可。"""
    lam = np.asarray(lambda_raw, dtype=float)
    is_zero = lam < tol
    if rule == "lrtest":
        is_zero = is_zero | (np.asarray(pvalue, dtype=float) > alpha)
    elif rule != "point":
        raise ValueError(f"zero_rule 须为 {ZERO_RULES}")
    return np.where(is_zero, 0.0, lam)


def fit_window(a, b, pit="window"):
    """
    对一个窗口拟合 Clayton + Gumbel，返回原始结果（未判零）。

    pit="window"：a, b 是收益率，在窗口内做秩变换（主口径）
    pit="given"： a, b 已是 (0,1) 上的 PIT 值，直接使用
    """
    a, b = np.asarray(a, float), np.asarray(b, float)
    u, v = (rank_pit(a), rank_pit(b)) if pit == "window" else (a, b)
    c, g = ClaytonCopula().fit(u, v), GumbelCopula().fit(u, v)
    return {
        "lambda_D_raw": c.lambda_raw, "lambda_U_raw": g.lambda_raw,
        "p_D": c.pvalue, "p_U": g.pvalue,
        "clayton_theta": c.theta, "gumbel_theta": g.theta,
        "clayton_ll": c.loglik, "gumbel_ll": g.loglik,
        "ll_diff": c.loglik - g.loglik,
        # 正态得分相关系数：窗口内「整体相关强度」，用于匹配高斯基准
        "ns_corr": float(np.corrcoef(norm.ppf(_clip01(u)), norm.ppf(_clip01(v)))[0, 1]),
    }


def apply_zero_rule(roll, rule="point", alpha=0.05, tol=LAMBDA_ZERO_TOL):
    """按判零口径生成 lambda_D / lambda_U 列。不重新拟合，切换口径很便宜。"""
    out = roll.copy()
    out["lambda_D"] = zero_out(out.lambda_D_raw, out.p_D, rule, alpha, tol)
    out["lambda_U"] = zero_out(out.lambda_U_raw, out.p_U, rule, alpha, tol)
    return out


def rolling_tail_dependence(x, y, window=60, step=1, pit="window",
                            zero_rule="point", alpha=0.05):
    """
    滚动窗口内分别拟合 Clayton 与 Gumbel，各取 λ_D / λ_U。
    中金原文口径：60 日窗口、每日滚动、两者独立估计、不做模型选择。

    x, y：pit="window" 时传收益率；pit="given" 时传已变换好的 PIT 值。
    返回 DataFrame（index=窗口最后一天），含原始估计与按 zero_rule 判零后的 λ。
    """
    df = pd.concat([pd.Series(x), pd.Series(y)], axis=1).dropna()
    xs, ys, idx = df.iloc[:, 0].values, df.iloc[:, 1].values, df.index

    rows, dates = [], []
    for end in range(window, len(idx) + 1, step):
        rows.append(fit_window(xs[end - window:end], ys[end - window:end], pit))
        dates.append(idx[end - 1])
    roll = pd.DataFrame(rows, index=pd.DatetimeIndex(dates, name="date"))
    return apply_zero_rule(roll, zero_rule, alpha)

In [7]:
def gaussian_null(window=60, n_sims=4000, rho=None, seed=1):
    """
    在高斯 copula（尾部相关严格为 0）下模拟窗口，走与实测完全相同的
    「窗口内秩变换 → 拟合」流程，返回每次模拟的原始结果。

    rho=None   ρ 在 [-0.9, 0.9] 上均匀抽取，用于构造「给定窗口相关强度下的误报率曲线」
    rho=数值   固定相关系数
    rho=数组   每次从中有放回抽取（如传入实测的 ns_corr，使基准的相关分布与实测一致）
    """
    rng = np.random.default_rng(seed)
    pool = None if rho is None or np.isscalar(rho) else np.clip(np.asarray(rho, float), -0.95, 0.95)
    rows = []
    for _ in range(n_sims):
        if rho is None:
            r = rng.uniform(-0.9, 0.9)
        elif pool is not None:
            r = rng.choice(pool)
        else:
            r = rho
        z1 = rng.standard_normal(window)
        z2 = r * z1 + np.sqrt(1 - r ** 2) * rng.standard_normal(window)
        res = fit_window(z1, z2, pit="window")
        res["rho_true"] = r
        rows.append(res)
    return pd.DataFrame(rows)


def noise_baseline(window=60, n_rep=300, rho=0.0, seed=1, zero_rule="point", alpha=0.05):
    """固定 ρ 的高斯基准下，三个判别量的命中率。ρ=0 即纯独立数据。"""
    sim = apply_zero_rule(gaussian_null(window, n_rep, rho, seed), zero_rule, alpha)
    ld, lu = sim.lambda_D > 0, sim.lambda_U > 0
    return {
        "rho": rho, "判零口径": zero_rule,
        "lambda_D>0占比": ld.mean(),
        "lambda_U>0占比": lu.mean(),
        "中金条件命中率": (ld & ~lu).mean(),
    }


def expected_under_gaussian(roll, null, zero_rule="point", alpha=0.05, bin_width=0.05):
    """
    对实测的每个窗口，按其正态得分相关系数 ns_corr 查高斯基准下的命中概率。

    返回 DataFrame（与 roll 同 index），三列分别是在
    「相同整体相关强度、但无尾部相关」的世界里，该窗口出现
    λ_D>0 / λ_U>0 / 中金条件 的概率。对时间取平均即为匹配基准。
    """
    sim = apply_zero_rule(null, zero_rule, alpha)
    edges = np.arange(-1.0, 1.0 + bin_width, bin_width)
    sim_bin = np.digitize(sim.ns_corr, edges)
    table = pd.DataFrame({
        "lambda_D>0": sim.lambda_D > 0,
        "lambda_U>0": sim.lambda_U > 0,
        "中金条件": (sim.lambda_D > 0) & (sim.lambda_U == 0),
    }).groupby(sim_bin).mean()
    # 空箱用相邻箱插值
    table = table.reindex(range(len(edges) + 1)).interpolate(limit_direction="both")
    return table.reindex(np.digitize(roll.ns_corr, edges)).set_axis(roll.index)

## 3. t-Copula 对高斯的检验

```text
椭圆族 Copula：高斯与 t。用来回答「有没有超出普通相关的尾部联动」。

══ 为什么用 t 对比高斯 ═══════════════════════════════════════════════

t-Copula 有两个参数，分工明确：
    ρ  整体相关强度（与高斯 Copula 的 ρ 含义相同）
    ν  自由度，管尾部厚度：ν 越小尾部越厚，ν→∞ 退化为高斯（尾部相关为 0）

所以「t 的似然是否显著好于高斯」恰好检验：在同样的整体相关下，
极端值是否比高斯所暗示的更容易同时出现。这是 Clayton 单参数做不到的。

══ 解读时的两个注意 ══════════════════════════════════════════════

1. t 的尾部相关系数 λ = 2·t_{ν+1}(−√((ν+1)(1−ρ)/(1+ρ))) 也随 ρ 增大，
   不能只看 λ 的数值，要看 ν 是否显著有限（即 LR 检验）。
2. t 的上下尾对称，且 ρ≈0 时四个角都会变厚。共同的波动率冲击
   （两个市场同一天都大幅波动、方向随机）也会让 ν 变小。
   所以 ν 显著有限说明「极端值倾向于同日出现」，不等于「倾向于同日下跌」。
   方向性要用 SJC 等非对称模型另行检验。
```

In [8]:
NU_MAX = 200.0       # ν 上限：超过即视同高斯


NU_MIN = 2.05


def gaussian_loglik(rho, u, v):
    x, y = norm.ppf(u), norm.ppf(v)
    det = 1 - rho ** 2
    return np.sum(-0.5 * np.log(det) - (rho ** 2 * (x ** 2 + y ** 2) - 2 * rho * x * y) / (2 * det))


def t_loglik(rho, nu, u, v):
    x, y = student_t.ppf(u, nu), student_t.ppf(v, nu)
    det = 1 - rho ** 2
    q = (x ** 2 - 2 * rho * x * y + y ** 2) / (nu * det)
    log_joint = (gammaln((nu + 2) / 2) - gammaln(nu / 2) - np.log(nu * np.pi)
                 - 0.5 * np.log(det) - (nu + 2) / 2 * np.log1p(q))
    # 两个一元 t 密度之积（归一化常数计两次）
    log_marg = (2 * (gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(nu * np.pi))
                - (nu + 1) / 2 * (np.log1p(x ** 2 / nu) + np.log1p(y ** 2 / nu)))
    return np.sum(log_joint - log_marg)


def t_tail_dependence(rho, nu):
    """t-Copula 的尾部相关系数（上下尾相同）。"""
    if not np.isfinite(nu) or nu >= NU_MAX:
        return 0.0
    return float(2 * student_t.cdf(-np.sqrt((nu + 1) * (1 - rho) / (1 + rho)), nu + 1))


def fit_gaussian(u, v):
    u, v = _clip01(u), _clip01(v)
    res = minimize_scalar(lambda r: -gaussian_loglik(r, u, v), bounds=(-0.99, 0.99), method="bounded")
    return float(res.x), float(-res.fun)


def fit_t(u, v):
    """
    极大似然估计 (ρ, ν)。ρ 的初值取 Kendall τ 的椭圆族换算 sin(πτ/2)，
    ν 从厚尾、薄尾两个初值出发，取似然更高者，避免停在局部解。
    """
    u, v = _clip01(u), _clip01(v)
    tau = kendalltau(u, v)[0]
    rho0 = float(np.clip(np.sin(np.pi * tau / 2), -0.95, 0.95))
    best = None
    for nu0 in (6.0, 60.0):
        res = minimize(lambda p: -t_loglik(p[0], p[1], u, v), x0=[rho0, nu0],
                       bounds=[(-0.99, 0.99), (NU_MIN, NU_MAX)], method="L-BFGS-B")
        if best is None or res.fun < best.fun:
            best = res
    rho, nu = map(float, best.x)
    return rho, nu, float(-best.fun)


def t_vs_gaussian(u, v):
    """
    似然比检验 H0：高斯（ν=∞） vs H1：t（ν 有限）。

    ν=∞ 在参数空间边界上，LR 的零分布是 ½χ²(0)+½χ²(1)，
    故 p = ½·P(χ²(1) ≥ LR)。test_t_lr_size_under_gaussian 检验了这一校准。
    """
    rho_g, ll_g = fit_gaussian(u, v)
    rho_t, nu, ll_t = fit_t(u, v)
    lr = max(2 * (ll_t - ll_g), 0.0)
    p = 0.5 * chi2.sf(lr, 1) if lr > 0 else 1.0
    return {"rho": rho_g, "rho_t": rho_t, "nu": nu, "lr": lr, "p_t": p,
            "lambda_t": t_tail_dependence(rho_t, nu)}


def rolling_t_test(x, y, window=250, step=1):
    """
    滚动窗口内（窗口内秩变换）做 t vs 高斯检验。
    x, y 为收益率；返回 DataFrame（index = 窗口最后一天）。
    """
    df = pd.concat([pd.Series(x), pd.Series(y)], axis=1).dropna()
    xs, ys, idx = df.iloc[:, 0].values, df.iloc[:, 1].values, df.index
    rows, dates = [], []
    for end in range(window, len(idx) + 1, step):
        a, b = xs[end - window:end], ys[end - window:end]
        rows.append(t_vs_gaussian(rank_pit(a), rank_pit(b)))
        dates.append(idx[end - 1])
    return pd.DataFrame(rows, index=pd.DatetimeIndex(dates, name="date"))

In [9]:
def empirical_tail_coexceedance(x, y, q=0.10):
    """
    非参数：股票落在最差 q 分位的日子里，债券也落在最差 q 分位的比例（下尾），
    以及对应的上尾比例。用于不依赖模型的交叉验证；高斯/独立时约等于 q 附近。
    """
    u, v = rank_pit(np.asarray(x)), rank_pit(np.asarray(y))
    lo, hi = u <= q, u > 1 - q
    return {"下尾同落": float((v[lo] <= q).mean()), "上尾同落": float((v[hi] > 1 - q).mean())}


def gaussian_null_t_test(window=250, n_sims=500, rho=0.0, seed=1):
    """高斯数据下 t-vs-高斯 检验的拒绝率（检验的名义水平校准用）。"""
    rng = np.random.default_rng(seed)
    pool = None if np.isscalar(rho) else np.asarray(rho, float)
    out = []
    for _ in range(n_sims):
        r = rng.choice(pool) if pool is not None else rho
        z1 = rng.standard_normal(window)
        z2 = r * z1 + np.sqrt(1 - r ** 2) * rng.standard_normal(window)
        out.append(t_vs_gaussian(rank_pit(z1), rank_pit(z2)))
    return pd.DataFrame(out)

## 4. SJC Copula 与上下尾不对称检验

```text
对称化 Joe-Clayton（SJC）Copula —— Patton (2006, International Economic Review)。

══ 为什么用 SJC ═══════════════════════════════════════════════════

中金分别拟合 Clayton（只有下尾）和 Gumbel（只有上尾），两个模型各估各的，
无法回答「下尾是否强于上尾」。SJC 在同一个模型里有两个参数，
直接就是两个尾部相关系数：

    τ_L = 下尾相关系数，τ_U = 上尾相关系数

「下尾强于上尾」因此可以写成一个正式的似然比检验：
    H0: τ_L = τ_U（对称）  vs  H1: τ_L ≠ τ_U，  LR ~ χ²(1)

⚠ 与 Clayton 一样，SJC 只能描述正相依；整体相关越强，τ_L、τ_U 都越大。
所以 τ 的绝对水平仍然混有普通相关，**看差值（不对称检验）比看水平可靠**。
负相关时两者都落到 0，此时不对称检验无意义（返回 NaN）。

══ 旋转：看「股跌债涨」 ═══════════════════════════════════════════

把第二个变量翻转（v → 1−v）后再拟合，原来的「左上角」变成「左下角」：
    τ_L(u, 1−v) = 股票极端下跌时，债券同时极端上涨的概率（避险 / flight-to-quality）
    τ_U(u, 1−v) = 股票极端上涨时，债券同时极端下跌的概率
```

In [10]:
TAU_MIN, TAU_MAX = 1e-4, 0.95     # 上限 0.95：再大 κ 会大到 (1−u)^κ 下溢；股债尾部相关远小于此


def _log1mexp(a):
    """log(1 − e^a)，a<0。a 接近 0 时用 expm1、远离 0 时用 log1p，两端都不丢精度。"""
    a = np.asarray(a, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(a > -0.6931, np.log(-np.expm1(a)), np.log1p(-np.exp(a)))


def jc_logpdf(u, v, tau_u, tau_l):
    """
    Joe-Clayton（BB7）copula 的对数密度。κ = 1/log2(2−τ_U)，γ = −1/log2(τ_L)。

    记 x = 1−(1−u)^κ，y = 1−(1−v)^κ，A = x^−γ + y^−γ − 1，h = A^(−1/γ)，w = 1−h，
    C = 1 − w^(1/κ)。对 u、v 求二阶混合偏导后整理得：
        c = a(u)·b(v)/κ · w^(1/κ−2) · A^(−1/γ−2) · [(1−1/κ)·h + (1+γ)·w]
    其中 a(u) = κ(1−u)^(κ−1)·x^(−γ−1)，b(v) 同理。全部在对数空间计算。
    """
    k = 1.0 / np.log2(2.0 - tau_u)
    g = -1.0 / np.log2(tau_l)
    lu, lv = np.log1p(-u), np.log1p(-v)                  # log(1−u), log(1−v)
    lx = _log1mexp(k * lu)                               # log x，x = 1 − (1−u)^κ
    ly = _log1mexp(k * lv)
    # log A，A = x^−γ + y^−γ − 1。右上角（u,v→1）时 A 只比 1 大一点点，
    # 直接相减会丢精度让 w=0、似然变 +inf，所以用 log1p(expm1+expm1)；
    # 左下角 A 极大时 expm1 会溢出，改用 logaddexp。两者按大小切换。
    ex, ey = -g * lx, -g * ly
    with np.errstate(over="ignore", invalid="ignore"):
        near_one = np.log1p(np.expm1(ex) + np.expm1(ey))
        s = np.logaddexp(ex, ey)
        far = s + np.log1p(-np.exp(-s))
    log_A = np.where(np.maximum(ex, ey) < 30, near_one, far)
    log_h = -log_A / g
    log_w = _log1mexp(log_h)
    h, w = np.exp(log_h), np.exp(log_w)
    log_a = np.log(k) + (k - 1) * lu + (-g - 1) * lx
    log_b = np.log(k) + (k - 1) * lv + (-g - 1) * ly
    return (log_a + log_b - np.log(k) + (1 / k - 2) * log_w + (-1 / g - 2) * log_A
            + np.log((1 - 1 / k) * h + (1 + g) * w))


def sjc_logpdf(u, v, tau_u, tau_l):
    """SJC = ½[JC(u,v; τU,τL) + JC(1−u,1−v; τL,τU)]，上下尾相关系数恰为 τ_U、τ_L。"""
    return np.log(0.5) + np.logaddexp(jc_logpdf(u, v, tau_u, tau_l),
                                      jc_logpdf(1 - u, 1 - v, tau_l, tau_u))


def fit_sjc(u, v, symmetric=False):
    """极大似然估计。symmetric=True 时约束 τ_L = τ_U（不对称检验的原假设）。"""
    u, v = _clip01(u), _clip01(v)
    if symmetric:
        res = minimize_scalar(lambda t: -np.sum(sjc_logpdf(u, v, t, t)),
                              bounds=(TAU_MIN, TAU_MAX), method="bounded")
        return float(res.x), float(res.x), float(-res.fun)
    best = None
    for start in ([0.1, 0.1], [0.3, 0.05], [0.05, 0.3]):
        res = minimize(lambda p: -np.sum(sjc_logpdf(u, v, p[0], p[1])), x0=start,
                       bounds=[(TAU_MIN, TAU_MAX)] * 2, method="L-BFGS-B")
        if best is None or res.fun < best.fun:
            best = res
    tau_u, tau_l = map(float, best.x)
    return tau_u, tau_l, float(-best.fun)


def asymmetry_test(u, v, min_tau=0.01):
    """
    检验 H0: τ_L = τ_U。返回两尾估计、LR 统计量与 p 值。
    对称约束下的估计也落在 0 附近时（整体负相关或不相关），没有尾部可比，p 值记为 NaN。
    """
    tau_u, tau_l, ll = fit_sjc(u, v)
    tau_s, _, ll_sym = fit_sjc(u, v, symmetric=True)
    lr = max(2 * (ll - ll_sym), 0.0)
    valid = max(tau_u, tau_l) >= min_tau
    return {"tau_L": tau_l, "tau_U": tau_u, "tau_sym": tau_s, "lr": lr,
            "p_asym": float(chi2.sf(lr, 1)) if valid else np.nan}

In [11]:
def rolling_asymmetry(u_returns, v_returns, window=250, step=5, flip_v=False):
    """滚动窗口（窗口内秩变换）的 SJC 不对称检验。flip_v=True 时检验旋转后的 (u, 1−v)。"""

    df = pd.concat([pd.Series(u_returns), pd.Series(v_returns)], axis=1).dropna()
    xs, ys, idx = df.iloc[:, 0].values, df.iloc[:, 1].values, df.index
    rows, dates = [], []
    for end in range(window, len(idx) + 1, step):
        u, v = rank_pit(xs[end - window:end]), rank_pit(ys[end - window:end])
        rows.append(asymmetry_test(u, 1 - v if flip_v else v))
        dates.append(idx[end - 1])
    return pd.DataFrame(rows, index=pd.DatetimeIndex(dates, name="date"))


def gaussian_tau_baseline(rhos, window=250, n_sims=100, seed=1):
    """
    高斯 Copula（尾部相关为 0、上下尾对称）下，SJC 估出的 τ 平均有多大。

    SJC 的 τ 水平混有普通相关：ρ=0.3 的高斯数据也会估出 τ≈0.12。
    用这张表扣掉「仅由相关强度带来的 τ」，剩下的才是超出普通相关的尾部联动。
    返回 DataFrame，index=ρ，列为 τ_L、τ_U 的均值及不对称检验 5% 水平的拒绝率。
    """

    rng = np.random.default_rng(seed)
    rows = {}
    for rho in rhos:
        res = []
        for _ in range(n_sims):
            z1 = rng.standard_normal(window)
            z2 = rho * z1 + np.sqrt(1 - rho ** 2) * rng.standard_normal(window)
            res.append(asymmetry_test(rank_pit(z1), rank_pit(z2)))
        res = pd.DataFrame(res)
        rows[rho] = {"tau_L均值": res.tau_L.mean(), "tau_U均值": res.tau_U.mean(),
                     "不对称检验误报率": (res.p_asym < 0.05).mean()}
    return pd.DataFrame(rows).T.rename_axis("rho")

## 5. 趋势与预警状态机

```text
趋势指标、预警状态机、预警事件切分与核对。
```

In [12]:
def dif_trend(px, fast=12, slow=26, kind="ma"):
    """
    DIF = 快线 - 慢线。>0 上涨趋势，<0 下跌趋势。

    中金原文写的是「标准 DIF 值（12 日移动平均 − 26 日移动平均）」，
    默认按字面用简单移动平均；kind="ema" 是 MACD 惯用的指数均线，作稳健性对照。
    """
    s = pd.Series(px).dropna()
    if kind == "ma":
        return s.rolling(fast).mean() - s.rolling(slow).mean()
    if kind == "ema":
        return s.ewm(span=fast, adjust=False).mean() - s.ewm(span=slow, adjust=False).mean()
    raise ValueError("kind 须为 'ma' 或 'ema'")


def warning_state(lambda_D, lambda_U, dif_a, dif_b, tol=LAMBDA_ZERO_TOL):
    """
    中金股债双杀预警状态机（dif_b 为债券）。

    触发：λ_D>0 且 λ_U=0 且（股或债 DIF<0）
    解除：λ_D=0 且 债券 DIF>0

    所有输入须在同一 index 上；返回 DataFrame，含两个条件与「预警中」状态。
    状态在 t 日收盘后才确定，交易上只能在 t+1 日使用。
    """
    sig = pd.DataFrame({"lambda_D": lambda_D, "lambda_U": lambda_U,
                        "DIF_a": dif_a, "DIF_b": dif_b})
    ld_pos = sig.lambda_D >= tol
    lu_zero = sig.lambda_U < tol
    sig["触发条件"] = ld_pos & lu_zero & ((sig.DIF_a < 0) | (sig.DIF_b < 0))
    sig["解除条件"] = ~ld_pos & (sig.DIF_b > 0)

    state, states = False, []
    for trig, rel in zip(sig["触发条件"], sig["解除条件"]):
        if not state and trig:
            state = True
        elif state and rel:
            state = False
        states.append(state)
    sig["预警中"] = states
    return sig


def episodes(state):
    """把布尔状态序列切成 [起, 止] 事件区间。持续交易日按 state 的 index 计数。"""
    state = pd.Series(state).fillna(False).astype(bool)
    blocks = (state != state.shift()).cumsum()
    rows = [(g.index[0], g.index[-1], len(g)) for _, g in state[state].groupby(blocks[state])]
    out = pd.DataFrame(rows, columns=["触发", "解除", "持续交易日"])
    return out.astype({"触发": "datetime64[ns]", "解除": "datetime64[ns]", "持续交易日": int})


# 中金原文披露的 5 次预警（沪深300 × 中债综合财富，2004/04–2020/12）
# 2008 年那次原文自认为错误信号。
CICC_EPISODES = pd.DataFrame({
    "触发": pd.to_datetime(["2004-10-20", "2006-06-14", "2008-09-12", "2013-06-20", "2016-12-12"]),
    "解除": pd.to_datetime(["2004-12-06", "2006-08-16", "2008-09-24", "2014-01-15", "2017-03-27"]),
    "原文判定": ["有效", "有效", "错误信号", "有效（钱荒）", "有效（钱荒）"],
})

In [13]:
def match_episodes(ours, reference=CICC_EPISODES, tolerance_days=20):
    """
    逐条核对：原文每次预警，复现里是否在 ±tolerance_days 个自然日内也有触发。
    另列出复现中多出来的（原文没有的）预警。
    """
    rows, used = [], set()
    for _, ref in reference.iterrows():
        gap = (ours["触发"] - ref["触发"]).dt.days
        near = gap[gap.abs() <= tolerance_days]
        if len(near):
            i = near.abs().idxmin()
            used.add(i)
            o = ours.loc[i]
            rows.append({**ref.to_dict(), "复现触发": o["触发"], "复现解除": o["解除"],
                         "触发偏差(天)": int(gap[i]), "复现持续交易日": o["持续交易日"]})
        else:
            rows.append({**ref.to_dict(), "复现触发": pd.NaT, "复现解除": pd.NaT,
                         "触发偏差(天)": None, "复现持续交易日": None})
    matched = pd.DataFrame(rows)
    extra = ours.drop(index=list(used))
    return matched, extra


def pair_warning(prices, returns, a, b, release_on, window=60, zero_rule="point", dif_kind="ma"):
    """
    任意两资产的「双杀」预警（中金：股债、股金、债金用完全相同的方法定义）。

    触发：λ_D>0 且 λ_U=0 且 a、b 至少一方 DIF<0
    解除：λ_D=0 且 release_on 的 DIF>0（股债预警里是债券；原文未说明其他两对用哪一方）
    返回与 warning_state 相同的 DataFrame，index 为重估日。
    """

    roll = rolling_tail_dependence(returns[a], returns[b], window=window, zero_rule=zero_rule)
    dif = {k: dif_trend(prices[k], kind=dif_kind).reindex(roll.index) for k in (a, b)}
    other = b if release_on == a else a
    # warning_state 的第 4 个参数是解除时要求上涨的资产
    return warning_state(roll.lambda_D, roll.lambda_U, dif[other], dif[release_on])

## 6. 日频回测与绩效

```text
日频回测：风险平价基准、预警叠加、绩效统计。

所有回测都在**日频**上做。信号若是隔 step 天重估的，先 reindex 到日频再前向填充，
不能直接在稀疏的重估点上累乘收益（那样只复利了 1/step 的交易日）。
```

In [14]:
TRADING_DAYS = 244


def to_daily(signal, daily_index):
    """把稀疏重估点上的信号扩展到日频：每天取最近一次重估的结果。"""
    return pd.Series(signal).reindex(daily_index, method="ffill")


def risk_parity_weights(returns, lookback=60, target_vol=0.08, max_leverage=None,
                        rebalance="ME"):
    """
    月末调仓的风险平价权重（中金原文：月末调仓、目标波动 8%）。

    两资产时等风险贡献的解恰好是波动率倒数加权（与相关系数无关）；
    多资产时这里同样用波动率倒数（朴素风险平价）。
    再整体缩放到目标波动，max_leverage 限制总杠杆。

    权重在调仓日收盘计算，从下一交易日起生效（shift(1)）。
    原文没说明协方差回看期与杠杆上限，这两个参数需做敏感性检验。
    """
    rets = returns.dropna()
    vol = rets.rolling(lookback).std()
    rebal_dates = rets.groupby(rets.index.to_period(rebalance[0])).tail(1).index

    w = {}
    for t in rebal_dates:
        if vol.loc[t].isna().any():
            continue
        raw = 1 / vol.loc[t]
        raw = raw / raw.sum()
        cov = rets.loc[:t].tail(lookback).cov() * TRADING_DAYS
        port_vol = float(np.sqrt(raw @ cov @ raw))
        k = target_vol / port_vol if target_vol else 1.0
        if max_leverage is not None:
            k = min(k, max_leverage)
        w[t] = raw * k
    w = pd.DataFrame(w).T
    return w.reindex(rets.index, method="ffill").shift(1).dropna()


def portfolio_returns(weights, returns, cash_rate=0.02):
    """
    组合日收益。权重和不为 1 的部分按现金利率计息（>1 即借入资金，按同一利率付息）。
    月内权重视为每日再平衡到目标权重（简化，忽略漂移）。
    """
    w = weights.reindex(returns.index).dropna()
    r = returns.reindex(w.index)
    cash_w = 1 - w.sum(axis=1)
    return (w * r).sum(axis=1) + cash_w * cash_rate / TRADING_DAYS


def overlay_warning(base_returns, warning, cash_rate=0.02):
    """
    预警叠加：t 日收盘处于预警 → t+1 日全部转为货币资产。
    warning 为日频布尔序列（t 日收盘时的状态）。
    """
    in_warn = (warning.reindex(base_returns.index, fill_value=False).astype(bool)
               .shift(1, fill_value=False))
    return pd.Series(np.where(in_warn, cash_rate / TRADING_DAYS, base_returns),
                     index=base_returns.index)


def max_drawdown(r):
    nav = (1 + r.dropna()).cumprod()
    return float((nav / nav.cummax() - 1).min())


def perf_stats(r, rf=0.02):
    """
    绩效统计。「收益波动比」= 年化收益/年化波动，与中金原文同口径；
    但空仓期按现金利率计息、波动为 0，会抬高这个比值，
    所以另给扣除无风险利率 rf 后的「超额夏普」，比较策略时以它为准。
    """
    r = r.dropna()
    yrs = len(r) / TRADING_DAYS
    ann = (1 + r).prod() ** (1 / yrs) - 1
    vol = r.std() * np.sqrt(TRADING_DAYS)
    mdd = max_drawdown(r)
    return {"年化收益": ann, "年化波动": vol, "收益波动比": ann / vol if vol else np.nan,
            "超额夏普": (ann - rf) / vol if vol else np.nan,
            "最大回撤": mdd, "卡玛比率": ann / abs(mdd) if mdd else np.nan}

In [15]:
def yearly_table(returns_df):
    """逐年收益与年内最大回撤（与中金原文的分年表同口径）。"""
    out = {}
    for name, r in returns_df.items():
        g = r.dropna().groupby(r.dropna().index.year)
        out[(name, "年收益")] = g.apply(lambda x: (1 + x).prod() - 1)
        out[(name, "年最大回撤")] = g.apply(max_drawdown)
    return pd.DataFrame(out)


def pairwise_cash_overlay(returns, base_weights, warnings, cash_rate=0.02):
    """
    中金多资产回撤控制：每日统计两两预警的触发数 n（t 日收盘状态，t+1 日执行）。
        n=0   按基准权重配置
        n=1   触发预警的两个资产仓位转为货币，其余资产保持基准权重
        n≥2   100% 货币
    base_weights：{资产: 权重}；warnings：{(资产a, 资产b): 日频布尔序列}。
    基准权重视为每日再平衡（与 portfolio_returns 一致）。
    """
    idx = returns.index
    w0 = pd.Series(base_weights, dtype=float)
    states = {pair: s.reindex(idx, fill_value=False).astype(bool).shift(1, fill_value=False)
              for pair, s in warnings.items()}
    n = sum(s.astype(int) for s in states.values())
    weights = pd.DataFrame(np.tile(w0.values, (len(idx), 1)), index=idx, columns=w0.index)
    for pair, s in states.items():
        single = s & (n == 1)
        weights.loc[single, list(pair)] = 0.0
    weights.loc[n >= 2, :] = 0.0
    r = (weights * returns[w0.index]).sum(axis=1) + (1 - weights.sum(axis=1)) * cash_rate / TRADING_DAYS
    return r, n

## 7. 分层聚类配置、风险平价与尾部风险平价

```text
基于（尾部）相关的分层聚类配置。

══ 研究对象 ══════════════════════════════════════════════════════

整理稿记载的东北证券（2025）方法：
    1. 各资产 ES（GPD 或非参数）
    2. 两两 Clayton 下尾相关 λ_L → 距离 d = −ln(λ_L)
    3. Ward 法生成树状结构
    4. 沿树自下而上，逐层用两资产公式合成尾部风险：
         TR_{1+2} = (w1·TR1 + w2·TR2) · (1 − w1·w2·(2/π)·arctan(d) · 2·min(TR1,TR2)/(TR1+TR2))
    5. 在权重上下限约束下最小化组合 TR

⚠ 未能取得东北证券原文。上式取自整理稿，无法核对是否抄写准确；
以下细节原文（整理稿）也未说明，本模块的取法：
    - 公式中的 w1、w2：取两簇在本节点内的相对权重（w_A/(w_A+w_B)）
    - 两簇之间的距离 d：两簇成员两两 d 的平均（平均连接）
    - λ_L = 0 时 d = −ln(0) = ∞：λ_L 下限截断为 1e-4（arctan 已接近 π/2，即视为完全分散）

对照方法：等权、波动率倒数、最小方差、HRP（López de Prado 2016，相关距离或尾部距离）、
直接最小化历史 ES（Rockafellar & Uryasev 2000 线性规划）。
```

In [16]:
LAMBDA_FLOOR = 1e-4


def corr_distance(R):
    """López de Prado 的相关距离 √(½(1−ρ))，取值 [0, 1]。"""
    rho = R.corr().values
    return pd.DataFrame(np.sqrt(np.clip(0.5 * (1 - rho), 0, None)), R.columns, R.columns)


def lower_tail_matrix(R):
    """两两 Clayton 下尾相关 λ_L（窗口内秩变换，点估计）。对角为 1。"""
    cols = R.columns
    U = {c: rank_pit(R[c].values) for c in cols}
    lam = np.eye(len(cols))
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            lam[i, j] = lam[j, i] = ClaytonCopula().fit(U[cols[i]], U[cols[j]]).lambda_raw
    return pd.DataFrame(lam, cols, cols)


def tail_distance(lam, kind="log"):
    """
    由 λ_L 得到距离。kind="log"：整理稿的 −ln(λ_L)（λ_L 截断于 1e-4）；
    kind="linear"：文献常用的 1 − λ_L（如 HRP 的尾部相关版本）。
    """
    L = np.clip(np.asarray(lam, float), LAMBDA_FLOOR, 1.0)
    D = -np.log(L) if kind == "log" else 1.0 - L
    np.fill_diagonal(D, 0.0)
    return pd.DataFrame(D, lam.index, lam.columns)


def tree(D, method="ward"):
    """由距离矩阵生成 scipy linkage 矩阵。"""
    return linkage(squareform(np.asarray(D, float), checks=False), method=method)


def cophenetic_corr(Z1, Z2):
    """两棵树的同表型相关：两两资产在两棵树上合并高度的相关系数。"""
    from scipy.cluster.hierarchy import cophenet

    return float(np.corrcoef(cophenet(Z1), cophenet(Z2))[0, 1])


def two_asset_tr(w1, tr1, tr2, d):
    """整理稿记载的两资产尾部风险合成公式（w2 = 1 − w1）。"""
    w2 = 1.0 - w1
    shrink = w1 * w2 * (2 / np.pi) * np.arctan(d) * 2 * np.minimum(tr1, tr2) / (tr1 + tr2)
    return (w1 * tr1 + w2 * tr2) * (1 - shrink)


def tree_tr(w, tr, D, Z):
    """
    沿 linkage 树自下而上合成组合尾部风险。
    w、tr：各资产权重与单资产 TR（numpy 数组）；D：资产两两距离（numpy 数组）。
    返回根节点的 TR（对应归一化后的整体组合）。
    """
    n = len(w)
    members = {i: [i] for i in range(n)}
    weight = {i: w[i] for i in range(n)}
    risk = {i: tr[i] for i in range(n)}
    for k, (a, b, _, _) in enumerate(Z):
        a, b = int(a), int(b)
        wa, wb = weight[a], weight[b]
        tot = wa + wb
        d = D[np.ix_(members[a], members[b])].mean()
        node = n + k
        risk[node] = two_asset_tr(wa / tot if tot > 0 else 0.5, risk[a], risk[b], d)
        weight[node] = tot
        members[node] = members[a] + members[b]
    return float(risk[2 * n - 2])


def nesc_weights(R, q=0.95, lower=1e-4, upper=0.20, es_func=es_hist):
    """整理稿方法：Ward 树 + arctan 递归合成，在权重上下限内最小化组合 TR。"""
    lam = lower_tail_matrix(R)
    D = tail_distance(lam, "log")
    Z = tree(D, "ward")
    tr = np.array([es_func(R[c], q) for c in R.columns])
    Dv = D.values
    n = R.shape[1]
    res = minimize(lambda w: tree_tr(w, tr, Dv, Z), x0=np.full(n, 1 / n), method="SLSQP",
                   bounds=[(lower, upper)] * n, constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   options={"maxiter": 300, "ftol": 1e-10})
    return pd.Series(res.x / res.x.sum(), R.columns)


def equal_weights(R):
    return pd.Series(1 / R.shape[1], R.columns)

In [17]:
def inverse_vol(R):
    iv = 1 / R.std()
    return iv / iv.sum()


def min_variance(R, lower=0.0, upper=0.20):
    S = R.cov().values
    n = S.shape[0]
    res = minimize(lambda w: w @ S @ w, x0=np.full(n, 1 / n), method="SLSQP",
                   bounds=[(lower, upper)] * n, constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   options={"maxiter": 300, "ftol": 1e-14})
    return pd.Series(res.x / res.x.sum(), R.columns)


def min_cvar(R, q=0.95, lower=0.0, upper=0.20):
    """
    直接最小化历史 ES（Rockafellar & Uryasev 2000）：
        min α + 1/((1−q)T)·Σ z_t   s.t.  z_t ≥ −r_t·w − α，z_t ≥ 0，Σw = 1，lower ≤ w ≤ upper
    """
    X = R.values
    T, n = X.shape
    c = np.concatenate([np.zeros(n), [1.0], np.full(T, 1 / ((1 - q) * T))])
    A_ub = np.hstack([-X, -np.ones((T, 1)), -np.eye(T)])
    A_eq = np.concatenate([np.ones(n), [0.0], np.zeros(T)])[None, :]
    bounds = [(lower, upper)] * n + [(None, None)] + [(0, None)] * T
    res = linprog(c, A_ub=A_ub, b_ub=np.zeros(T), A_eq=A_eq, b_eq=[1.0], bounds=bounds, method="highs")
    return pd.Series(res.x[:n] / res.x[:n].sum(), R.columns)


def hrp_weights(R, D=None, method="single"):
    """
    López de Prado（2016）分层风险平价：按树的叶序做准对角化，
    再递归二分，每次按两半的逆方差组合方差反比分配权重。D 缺省时用相关距离。
    """
    D = corr_distance(R) if D is None else D
    order = list(R.columns[leaves_list(tree(D, method))])
    S = R.cov()

    def cluster_var(items):
        sub = S.loc[items, items].values
        ivp = 1 / np.diag(sub)
        ivp /= ivp.sum()
        return float(ivp @ sub @ ivp)

    w = pd.Series(1.0, index=order)
    clusters = [order]
    while clusters:
        nxt = []
        for c in clusters:
            if len(c) < 2:
                continue
            left, right = c[: len(c) // 2], c[len(c) // 2:]
            vl, vr = cluster_var(left), cluster_var(right)
            alpha = 1 - vl / (vl + vr)
            w[left] *= alpha
            w[right] *= 1 - alpha
            nxt += [left, right]
        clusters = nxt
    return w.reindex(R.columns)


def cap_weights(w, upper=0.20):
    """把超过上限的权重截到上限，超出部分按比例分给未触顶的资产，直到没有资产超限。"""
    w = pd.Series(w, dtype=float) / pd.Series(w, dtype=float).sum()
    for _ in range(len(w)):
        over = w > upper + 1e-12
        if not over.any():
            break
        excess = (w[over] - upper).sum()
        w[over] = upper
        free = ~over & (w < upper)
        w[free] += excess * w[free] / w[free].sum()
    return w

In [18]:
def block_bootstrap_sharpe_diff(r1, r2, rf=0.02, block=20, n_boot=2000, seed=0, periods=244):
    """
    两条日收益序列超额夏普之差的分块自助法置信区间（按连续日期块重抽样，保留波动率聚集等时间序列相关）。
    返回 (点估计, 2.5% 分位, 97.5% 分位, 差值 ≤ 0 的比例)。
    """
    x = pd.concat([r1, r2], axis=1).dropna().values
    T = len(x)
    rng = np.random.default_rng(seed)

    def sharpe(a):
        return (a.mean() * periods - rf) / (a.std() * np.sqrt(periods))

    point = sharpe(x[:, 0]) - sharpe(x[:, 1])
    diffs = []
    for _ in range(n_boot):
        starts = rng.integers(0, T - block, T // block + 1)
        idx = np.concatenate([np.arange(s, s + block) for s in starts])[:T]
        diffs.append(sharpe(x[idx, 0]) - sharpe(x[idx, 1]))
    diffs = np.array(diffs)
    return float(point), float(np.quantile(diffs, .025)), float(np.quantile(diffs, .975)), float((diffs <= 0).mean())


def rolling_backtest(returns, strategies, lookback=250, rebalance="M", start=None):
    """
    每月末用过去 lookback 个交易日的收益计算各策略权重，从下一交易日起持有到下次调仓
    （期间每日再平衡到目标权重）。返回 (日收益 DataFrame, {策略: 权重 DataFrame})。
    """
    R = returns.dropna()
    ends = R.groupby(R.index.to_period(rebalance)).tail(1).index
    ends = [t for t in ends if R.index.get_loc(t) + 1 >= lookback and (start is None or t >= pd.Timestamp(start))]
    weights = {name: {} for name in strategies}
    for t in ends:
        win = R.loc[:t].tail(lookback)
        for name, f in strategies.items():
            weights[name][t] = f(win)
    out, W = {}, {}
    for name in strategies:
        wd = pd.DataFrame(weights[name]).T
        wd = wd.reindex(R.index, method="ffill").shift(1).dropna()
        W[name] = pd.DataFrame(weights[name]).T
        out[name] = (wd * R.reindex(wd.index)).sum(axis=1)
    return pd.DataFrame(out), W


def turnover(W):
    """平均单边换手率（每次调仓权重变化绝对值之和的一半，按调仓次数平均）。"""
    return float(W.diff().abs().sum(axis=1).iloc[1:].mean() / 2)


def erc_weights(R, budgets=None):
    """普通风险平价：各资产对组合波动率的贡献相等（或按 budgets）。"""
    import cvxpy as cp

    S = R.cov().values
    n = S.shape[0]
    b = np.full(n, 1 / n) if budgets is None else np.asarray(budgets, float)
    scale = 1 / np.sqrt(np.diag(S).mean())                     # 数值缩放，不影响解
    y = cp.Variable(n, pos=True)
    cp.Problem(cp.Minimize(0.5 * cp.quad_form(y, cp.psd_wrap(S * scale ** 2)) - b @ cp.log(y))).solve(solver="CLARABEL")
    w = np.maximum(y.value, 0)
    return pd.Series(w / w.sum(), R.columns)

In [19]:
def es_parity_weights(scenarios, q=0.95, budgets=None):
    """
    尾部风险平价：各资产对组合 ES 的贡献相等（或按 budgets）。
    scenarios：T×n 收益情景（历史收益，或 GARCH 过滤后的情景）。ES 用 Rockafellar–Uryasev 表达式。
    """
    import cvxpy as cp

    X = np.asarray(scenarios, float)
    X = X / X.std(axis=0).mean()                                  # 数值缩放，不影响解
    T, n = X.shape
    b = np.full(n, 1 / n) if budgets is None else np.asarray(budgets, float)
    y = cp.Variable(n, pos=True)
    alpha = cp.Variable()
    u = cp.Variable(T, nonneg=True)                               # u_t ≥ 组合损失超过 α 的部分
    prob = cp.Problem(cp.Minimize(alpha + cp.sum(u) / ((1 - q) * T) - b @ cp.log(y)), [u >= -X @ y - alpha])
    try:
        prob.solve(solver="CLARABEL")                             # 情景数在一千左右时快而准
    except cp.error.SolverError:
        prob.solve(solver="SCS", eps=1e-8, max_iters=200_000)     # 情景更多时 Clarabel 会失败
    w = np.maximum(y.value, 0)
    cols = scenarios.columns if hasattr(scenarios, "columns") else range(n)
    return pd.Series(w / w.sum(), cols)


def es_contributions(w, scenarios, q=0.95):
    """
    组合 ES 的欧拉分解：RC_i = −w_i · E[r_i | 组合落在最差的 (1−q) 尾部]，各项之和等于组合 ES。
    """
    X = np.asarray(scenarios, float)
    w = np.asarray(w, float)
    port = X @ w
    k = max(int(np.ceil((1 - q) * len(port))), 1)
    tail = np.argsort(port)[:k]
    return -w * X[tail].mean(axis=0)


def vol_contributions(w, R):
    S = np.asarray(R.cov(), float)
    w = np.asarray(w, float)
    return w * (S @ w) / np.sqrt(w @ S @ w)


def fhs_scenarios(R):
    """
    GARCH 过滤的历史情景：各资产分别拟合 GJR-GARCH(1,1)，得到标准化残差 z_t，
    再乘以各自明日的预测波动率。保留历史上各资产残差的同日联动（依赖结构），
    但把尺度换成当前的波动率水平（Barone-Adesi 等 1999 的多元 FHS）。
    """

    out = {}
    for c in R.columns:
        x = R[c].values
        p = fit_gjr(x)
        sd, sd_next = gjr_filter(x, p)
        out[c] = (x - p["mu"] / 100) / sd * sd_next + p["mu"] / 100
    return pd.DataFrame(out, index=R.index)

## 8. KDE 模拟（国泰君安）

```text
保留边缘分布的多资产收益模拟：核密度（KDE）边缘 + 三种注入相关的方法。

══ 研究对象 ══════════════════════════════════════════════════════

国泰君安（2018）《资产配置之步步为营：尾部风险控制与优化》（数量化专题之一百零九，
陈奥林、叶尔乐）。原文第 1.3 节：先对各资产历史收益做高斯 KDE、独立抽样得到 K；
再对历史协方差 Σ 做 Cholesky 分解 Σ = AAᵀ，用递推式注入相关：

    X_i = Σ_{j<i} (K_j − μ_j)/A_jj · A_ij + K_i

原文称「从公式的角度可以保证协方差一致」，并用逐年 KS 检验（1% 水平）验证边缘分布。两点都不成立：
    · Var(X_i) = Σ_{j<i} (A_ij/A_jj)² Var(K_j) + Var(K_i)，一般大于 Σ_ii；
    · X_i 是独立变量之和，其分布是卷积，偏度、峰度被稀释。
而且结果依赖资产的排列顺序：排在第一位的资产不受影响，越靠后的被改动越多。
（原文 3.3 节自己也承认「对相关性的捕捉方法只是近似的方法」。）

两种能严格保留边缘分布的做法，作为对照：
    gaussian_copula_kde   高斯 Copula 生成相依的均匀变量，再用各资产 KDE 的分位数函数逆变换
    iman_conover          Iman-Conover 秩重排：保留每列抽样值不变，只重排顺序以注入目标秩相关
```

In [20]:
def kde_bandwidth(x):
    """Silverman 经验带宽。"""
    x = np.asarray(x, float)
    s = min(x.std(ddof=1), (np.quantile(x, .75) - np.quantile(x, .25)) / 1.349)
    return 0.9 * s * len(x) ** (-0.2)


def kde_sample(x, n, rng):
    """从高斯 KDE 精确抽样：随机取一个历史点，再加上 N(0, h²) 噪声。"""
    x = np.asarray(x, float)
    return rng.choice(x, n) + kde_bandwidth(x) * rng.standard_normal(n)


def kde_ppf(x, u, grid_size=4000):
    """高斯 KDE 的分位数函数（在细网格上数值求逆 CDF）。"""
    x = np.asarray(x, float)
    h = kde_bandwidth(x)
    grid = np.linspace(x.min() - 6 * h, x.max() + 6 * h, grid_size)
    cdf = norm.cdf((grid[:, None] - x[None, :]) / h).mean(axis=1)
    return np.interp(u, cdf, grid)


def gtja_recursion(hist, n, rng):
    """整理稿记载的国泰君安递推式。"""
    H = np.asarray(hist, float)
    k = H.shape[1]
    K = np.column_stack([kde_sample(H[:, j], n, rng) for j in range(k)])
    A = np.linalg.cholesky(np.cov(H, rowvar=False))
    mu = H.mean(axis=0)
    X = K.copy()
    for i in range(1, k):
        for j in range(i):
            X[:, i] += (K[:, j] - mu[j]) / A[j, j] * A[i, j]
    return X


def gaussian_copula_kde(hist, n, rng):
    """正态得分相关 → 多元正态 → Φ 得到相依的均匀变量 → 各资产 KDE 分位数逆变换。"""
    H = np.asarray(hist, float)
    T, k = H.shape
    Zs = norm.ppf(np.column_stack([rankdata(H[:, j]) / (T + 1) for j in range(k)]))
    C = np.corrcoef(Zs, rowvar=False)
    U = norm.cdf(rng.multivariate_normal(np.zeros(k), C, n))
    return np.column_stack([kde_ppf(H[:, j], U[:, j]) for j in range(k)])


def iman_conover(hist, n, rng):
    """
    Iman & Conover (1982)：各列独立地从 KDE 抽样，再按目标秩相关重排各列顺序。
    目标相关取历史收益的正态得分相关，与 gaussian_copula_kde 一致。
    """
    H = np.asarray(hist, float)
    T, k = H.shape
    K = np.column_stack([np.sort(kde_sample(H[:, j], n, rng)) for j in range(k)])
    Zs = norm.ppf(np.column_stack([rankdata(H[:, j]) / (T + 1) for j in range(k)]))
    C = np.corrcoef(Zs, rowvar=False)
    # 得分矩阵：van der Waerden 得分的随机排列，去相关后再乘目标相关的 Cholesky 因子
    S = np.column_stack([rng.permutation(norm.ppf(np.arange(1, n + 1) / (n + 1))) for _ in range(k)])
    E = np.corrcoef(S, rowvar=False)
    S_star = S @ np.linalg.inv(np.linalg.cholesky(E)).T @ np.linalg.cholesky(C).T
    out = np.empty_like(K)
    for j in range(k):
        out[:, j] = K[:, j][rankdata(S_star[:, j], method="ordinal").astype(int) - 1]
    return out


METHODS = {"国泰君安递推": gtja_recursion, "高斯Copula+KDE": gaussian_copula_kde, "Iman-Conover": iman_conover}


def simulate_horizon(method, hist, horizon, n_paths, rng):
    """
    模拟 n_paths 条长度为 horizon 的日收益路径（各日独立同分布），返回 (n_paths, horizon, k)。
    """
    k = np.asarray(hist).shape[1]
    draws = method(hist, n_paths * horizon, rng)
    return draws.reshape(n_paths, horizon, k)

In [21]:
def simplex_grid(k, step):
    """k 个资产、步长 step 的全部非负权重组合（和为 1）。"""
    n = int(round(1 / step))
    out = []

    def rec(prefix, left, depth):
        if depth == k - 1:
            out.append(prefix + [left])
            return
        for i in range(left + 1):
            rec(prefix + [i], left - i, depth + 1)

    rec([], n, 0)
    return np.array(out, dtype=float) / n


def path_risk(paths, W, x=0.01, y=-0.01, chunk=400):
    """
    paths：(n_paths, T, k) 模拟日简单收益；W：(m, k) 候选权重（期初配比，期内不再平衡）。
    返回 dict：TR、IH、EH、期末对数收益均值，各为长度 m 的数组。
    """
    V = np.cumprod(1 + paths, axis=1).astype(np.float32)          # 各资产的价值路径
    n, T, k = V.shape
    flat = V.reshape(n * T, k)
    out = {key: [] for key in ("TR", "IH", "EH", "mean_X")}
    for s in range(0, len(W), chunk):
        Wc = W[s:s + chunk].astype(np.float32)
        P = (flat @ Wc.T).reshape(n, T, -1)                          # (n, T, m) 组合价值
        # 比较价值而不取对数：Y ≤ y ⟺ min P ≤ e^y，X ≤ x ⟺ P_T ≤ e^x（只对期末取对数）
        PT = P[:, -1, :]
        ih = P.min(axis=1) <= np.exp(y)                              # y ≤ 0，t=0 时 P_0=1 不触发
        eh = ~ih & (PT <= np.exp(x))
        out["IH"].append(ih.mean(0)); out["EH"].append(eh.mean(0))
        out["TR"].append((ih | eh).mean(0)); out["mean_X"].append(np.log(PT).mean(0))
    return {key: np.concatenate(v) for key, v in out.items()}


def gtja_optimal_weights(paths, grid, x=0.01, y=-0.01):
    """
    在权重网格上最小化总风险 TR。TR 是路径计数的比例、对权重不连续，原文未说明求解方法；
    这里用网格穷举，并列最小时取模拟期末对数收益均值最高的一组。
    """
    r = path_risk(paths, grid, x, y)
    best = np.flatnonzero(r["TR"] == r["TR"].min())
    i = best[np.argmax(r["mean_X"][best])]
    return grid[i], {key: float(v[i]) for key, v in r.items()}


def gtja_backtest(returns, method, window=120, horizon=20, n_paths=1500, step=0.05,
                  x=0.01, y=-0.01, start=None, seed=0, order=None):
    """
    原文的滚动回测：每 horizon 个交易日，用过去 window 日的收益模拟 n_paths 条 horizon 日路径，
    在权重网格上最小化 TR，按权重配比后持有 horizon 日（期内不再平衡）。
    order：资产在模拟时的排列顺序（只影响国泰君安递推式）。
    返回 (日收益 Series, 权重 DataFrame, 模拟风险 DataFrame)。
    """
    R = returns.dropna()
    cols = list(R.columns) if order is None else order
    rng = np.random.default_rng(seed)
    grid = simplex_grid(len(cols), step)
    first = window if start is None else max(window, R.index.searchsorted(pd.Timestamp(start)))
    daily, wrec, rrec = [], {}, {}
    for t0 in range(first, len(R) - 1, horizon):
        H = R[cols].iloc[t0 - window:t0].values
        w, risk = gtja_optimal_weights(simulate_horizon(method, H, horizon, n_paths, rng), grid, x, y)
        date = R.index[t0 - 1]
        wrec[date], rrec[date] = pd.Series(w, cols), risk
        hold = R[cols].iloc[t0:t0 + horizon]
        value = (1 + hold).cumprod() @ w                              # 期初配比、期内漂移
        daily.append(value.pct_change().fillna(value.iloc[0] - 1))
    return (pd.concat(daily), pd.DataFrame(wrec).T[list(R.columns)], pd.DataFrame(rrec).T)

## 9. 单资产 VaR / ES 滚动预测与回测

```text
单资产 VaR / ES 的滚动预测与回测。

══ 六种方法 ══════════════════════════════════════════════════════

    历史模拟（HS）     窗口内损失的经验分位数与超出部分均值
    正态              窗口内均值、标准差
    t 分布            窗口内极大似然拟合的 Student-t
    GPD（POT）        对窗口内超过 90% 分位的损失拟合广义帕累托分布
    GARCH-FHS         GJR-GARCH(1,1) 预测明日波动率 × 标准化残差的经验分位（过滤历史模拟）
    GARCH-EVT         GJR-GARCH(1,1) 预测明日波动率 × 标准化残差尾部的 GPD 分位
                      （McNeil & Frey 2000, Journal of Empirical Finance）

前四种不随波动率变化（无条件）；后两种随 GARCH 预测的波动率每日调整（条件）。
所有预测只用截至当日的数据，用来对次日收益做回测。损失记为正数。

══ 回测 ═════════════════════════════════════════════════════════

    Kupiec（1995）       VaR 突破频率是否等于 1 − 置信水平（无条件覆盖）
    Christoffersen（1998）突破是否扎堆（独立性）
    Acerbi–Szekely（2014）Z2 ES 检验：Z2 = Σ(X_t·I_t)/(T·α·ES_t) + 1，X 为收益；
                         原文给出 α=2.5% 时 5% 水平的临界值约 −0.70（< −0.70 即 ES 被低估）
```

In [22]:
def var_es_hist(r, q):
    loss = -np.asarray(r)
    var = np.quantile(loss, q)
    return var, loss[loss >= var].mean()


def var_es_normal(r, q):
    mu, sd = np.mean(r), np.std(r, ddof=1)
    z = norm.ppf(q)
    return -mu + sd * z, -mu + sd * norm.pdf(z) / (1 - q)


def var_es_t(r, q):
    nu, loc, scale = student_t.fit(r)
    nu = max(nu, 2.05)
    x = student_t.ppf(q, nu)
    es_std = student_t.pdf(x, nu) / (1 - q) * (nu + x ** 2) / (nu - 1)
    return -loc + scale * x, -loc + scale * es_std


def gpd_fit(loss, threshold=0.90):
    """对超过 threshold 分位的损失超额拟合 GPD，返回 (u, ξ, β, 样本数 n, 超额数 N_u)。"""
    loss = np.asarray(loss)
    u = np.quantile(loss, threshold)
    exc = loss[loss > u] - u
    xi, _, beta = genpareto.fit(exc, floc=0)
    return u, xi, beta, len(loss), len(exc)


def gpd_tail(loss, q, threshold=0.90, fit=None):
    """POT-GPD 的 VaR 与 ES（损失为正）。fit 可传入 gpd_fit 的结果以免重复拟合。"""
    u, xi, beta, n, nu = gpd_fit(loss, threshold) if fit is None else fit
    ratio = n / nu * (1 - q)
    if abs(xi) < 1e-6:
        var = u - beta * np.log(ratio)
        return var, var + beta
    var = u + beta / xi * (ratio ** (-xi) - 1)
    es = (var + beta - xi * u) / (1 - xi) if xi < 1 else np.inf
    return var, es


def var_es_gpd(r, q, fit=None):
    return gpd_tail(-np.asarray(r), q, fit=fit)


def fit_gjr(r):
    """正态 QMLE 拟合 GJR-GARCH(1,1)（收益单位为 %），返回参数 dict。"""
    from arch import arch_model

    res = arch_model(np.asarray(r) * 100, mean="Constant", vol="GARCH", p=1, o=1, q=1,
                     dist="normal").fit(disp="off", show_warning=False)
    p = res.params
    return {"mu": p["mu"], "omega": p["omega"], "alpha": p["alpha[1]"], "gamma": p["gamma[1]"], "beta": p["beta[1]"]}


def gjr_filter(r, params):
    """给定参数，对收益序列递推条件方差；返回 (各日 σ_t, 下一日 σ_{T+1})，单位为收益（非 %）。"""
    x = np.asarray(r) * 100 - params["mu"]
    var = np.empty(len(x) + 1)
    var[0] = x.var()
    for t in range(len(x)):
        neg = x[t] < 0
        var[t + 1] = (params["omega"] + (params["alpha"] + params["gamma"] * neg) * x[t] ** 2
                      + params["beta"] * var[t])
    sd = np.sqrt(var) / 100
    return sd[:-1], sd[-1]


UNCONDITIONAL = {"历史模拟": var_es_hist, "正态": var_es_normal, "t 分布": var_es_t, "GPD": var_es_gpd}

In [23]:
def rolling_forecasts(returns, window=1000, q_var=0.99, q_es=0.975, refit=21, start=None):
    """
    每个交易日收盘后，用过去 window 日的收益预测下一交易日的：
        VaR（q_var，默认 99%）、VaR_es（q_es，默认 97.5%）、ES（q_es）
    ES 回测要用同一置信水平的 VaR，所以另存 VaR_es。
    t 分布与 GARCH 参数每 refit 日重估一次（GARCH 波动率仍每日递推），其余方法每日重算。
    返回 DataFrame：index = 预测所针对的交易日，列为 (方法, 指标)，另有 ("收益", "")。
    """
    r = pd.Series(returns).dropna()
    vals, idx = r.values, r.index
    first = window if start is None else max(window, idx.searchsorted(pd.Timestamp(start)))
    rows, dates = [], []
    t_cache, g_params, z_q = None, None, None
    for i in range(first, len(vals)):
        w = vals[i - window:i]
        row = {}
        for name, f in UNCONDITIONAL.items():
            if name == "t 分布":
                if t_cache is None or (i - first) % refit == 0:
                    t_cache = (f(w, q_var), f(w, q_es))
                (v, _), (ve, e) = t_cache
            elif name == "GPD":
                fit = gpd_fit(-w)
                v, _ = f(w, q_var, fit=fit)
                ve, e = f(w, q_es, fit=fit)
            else:
                v, _ = f(w, q_var)
                ve, e = f(w, q_es)
            row[(name, "VaR")], row[(name, "VaR_es")], row[(name, "ES")] = v, ve, e
        if g_params is None or (i - first) % refit == 0:
            g_params = fit_gjr(w)
            sd_hist, _ = gjr_filter(w, g_params)
            z = (w - g_params["mu"] / 100) / sd_hist
            zl = -z
            zfit = gpd_fit(zl)
            z_q = {"FHS": (var_es_hist(z, q_var)[0], *var_es_hist(z, q_es)),
                   "EVT": (gpd_tail(zl, q_var, fit=zfit)[0], *gpd_tail(zl, q_es, fit=zfit))}
        _, sd_next = gjr_filter(w, g_params)
        mu = g_params["mu"] / 100
        for tag, (zv, zve, ze) in z_q.items():
            row[(f"GARCH-{tag}", "VaR")] = -mu + sd_next * zv
            row[(f"GARCH-{tag}", "VaR_es")] = -mu + sd_next * zve
            row[(f"GARCH-{tag}", "ES")] = -mu + sd_next * ze
        row[("收益", "")] = vals[i]
        rows.append(row)
        dates.append(idx[i])
    out = pd.DataFrame(rows, index=pd.DatetimeIndex(dates))
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    return out


def kupiec(hits, p):
    """无条件覆盖：突破频率是否等于 p。返回 (突破率, p 值)。"""
    hits = np.asarray(hits, bool)
    n, x = len(hits), hits.sum()
    pi = x / n
    if x in (0, n):
        lr = -2 * (x * np.log(p) + (n - x) * np.log(1 - p)) if x == 0 else np.inf
    else:
        lr = -2 * ((n - x) * np.log(1 - p) + x * np.log(p) - (n - x) * np.log(1 - pi) - x * np.log(pi))
    return float(pi), float(chi2.sf(lr, 1))

In [24]:
def christoffersen(hits):
    """独立性：今天是否突破与昨天是否突破无关。返回 (突破后再突破的概率, p 值)。"""
    h = np.asarray(hits, int)
    a, b = h[:-1], h[1:]
    n00, n01 = ((a == 0) & (b == 0)).sum(), ((a == 0) & (b == 1)).sum()
    n10, n11 = ((a == 1) & (b == 0)).sum(), ((a == 1) & (b == 1)).sum()
    p01 = n01 / max(n00 + n01, 1)
    p11 = n11 / max(n10 + n11, 1)
    p = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)

    def ll(k, n, prob):
        return 0.0 if prob in (0, 1) else k * np.log(prob) + (n - k) * np.log(1 - prob)

    lr = -2 * (ll(n01 + n11, n00 + n01 + n10 + n11, p) - ll(n01, n00 + n01, p01) - ll(n11, n10 + n11, p11))
    return float(p11), float(chi2.sf(max(lr, 0.0), 1))


def acerbi_szekely_z2(returns, var, es, alpha):
    """
    Z2 = Σ_t (X_t · I_t) / (T · α · ES_t) + 1，I_t = 1{X_t < −VaR_t}（VaR 与 ES 取同一 α）。
    期望为 0；显著为负说明 ES 被低估。
    """
    x, v, e = (np.asarray(a, float) for a in (returns, var, es))
    hit = x < -v
    return float(np.sum(x * hit / e) / (len(x) * alpha) + 1)


def backtest_table(fc, q_var=0.99, q_es=0.975):
    """
    各方法：99% VaR 的突破率与 Kupiec、Christoffersen 检验；
    97.5% ES 的 Acerbi–Szekely Z2，以及「97.5% VaR 突破日的平均实际损失 / 平均预测 ES」（>1 表示 ES 偏低）。
    """
    x = fc[("收益", "")].values
    methods = [m for m in fc.columns.get_level_values(0).unique() if m != "收益"]
    rows = {}
    for m in methods:
        hit = x < -fc[(m, "VaR")].values
        rate, p_k = kupiec(hit, 1 - q_var)
        p11, p_c = christoffersen(hit)
        ve, es = fc[(m, "VaR_es")].values, fc[(m, "ES")].values
        hit_e = x < -ve
        rows[m] = {"99% VaR 突破率": rate, "Kupiec p": p_k, "突破后次日再突破": p11, "Christoffersen p": p_c,
                   "Z2": acerbi_szekely_z2(x, ve, es, 1 - q_es),
                   "实际损失/预测ES": (-x[hit_e]).mean() / es[hit_e].mean() if hit_e.any() else np.nan}
    return pd.DataFrame(rows).T

## 10. 单资产尾部预警（兴业）

```text
单资产尾部风险预警 —— 兴业证券（2025）《基于宏观相似、动量趋势和尾部风险控制的全球资产配置研究》第 4 章。

══ 原文的信号 ═══════════════════════════════════════════════════════
（「超过过去 N 的 99% 分位数」「近五日斜率为正」以下分别记为 exceeds、rising）

    沪深300/中证500  股债性价比（全A股息率 − 10年国债收益率）低于过去 4 年的 10% 分位
                    且 指数近 5 日累计收益为负 且 PcaScore 近 21 日均值 > 近 63 日均值
    恒生指数         美元兑离岸人民币 exceeds(2 年) 且 美国 10 年实际利率 exceeds(1 年) 且 rising
    标普500         VIX exceeds(1 年) 且 rising  或  实际利率 exceeds(2 年) 且 rising
    纳斯达克100      同上，VIX 换成 VXN
    黄金            实际利率 exceeds(3 个月) 且 rising
    原油            OVX exceeds(1 年) 且 rising
    南华商品指数      美元指数 exceeds(2 年) 且 rising（免费数据不足，未复现）

触发后：事件研究看未来 1/2/3/5/10 日平均收益；择时策略在信号后 5 个交易日空仓。

══ 实现上的取法（原文未说明）════════════════════════════════════════

    · 分位数阈值只用截至前一日的数据（shift(1)），当日值与之比较
    · 「近五日斜率」取最近 5 个观测值线性回归的斜率
    · 1 年 / 2 年 / 3 个月 / 4 年 = 252 / 504 / 63 / 1008 个观测
    · PcaScore：窗口内各成分收益协方差矩阵前 2 个特征值之和 / 前 20 个之和
```

In [25]:
YEAR = 252


def exceeds(x, window, q=0.99):
    """当日值是否高于截至前一日、过去 window 个观测的 q 分位数。"""
    x = pd.Series(x).dropna()
    thr = x.rolling(window, min_periods=window).quantile(q).shift(1)
    return (x > thr) & thr.notna()


def below(x, window, q=0.10):
    """当日值是否低于截至前一日、过去 window 个观测的 q 分位数。"""
    x = pd.Series(x).dropna()
    thr = x.rolling(window, min_periods=window).quantile(q).shift(1)
    return (x < thr) & thr.notna()


def rising(x, n=5):
    """最近 n 个观测的线性回归斜率是否为正。"""
    x = pd.Series(x).dropna()
    t = np.arange(n) - (n - 1) / 2
    slope = x.rolling(n).apply(lambda v: np.dot(t, v - v.mean()), raw=True)
    return slope > 0


def pca_score(returns, window=21, top=2, total=20):
    """
    PcaScore：窗口内收益协方差矩阵的前 top 个特征值之和，占前 total 个之和的比例。
    数值越高，成分之间走势越趋同。
    """
    R = returns.dropna()
    X = R.values
    out = np.full(len(R), np.nan)
    for end in range(window, len(R) + 1):
        ev = np.sort(np.linalg.eigvalsh(np.cov(X[end - window:end], rowvar=False)))[::-1]
        out[end - 1] = ev[:top].sum() / ev[:total].sum()
    return pd.Series(out, R.index)


def align(signal, target_index, lag=0):
    """
    把信号对齐到目标资产的交易日：对目标日 d，取信号在「d − lag 个自然日」当天或之前最后一个值。
        lag=0  同日对齐（原文做法）
        lag=1  只用目标日之前的信号日 —— 美国收盘晚于亚洲收盘，美国 d 日的数据
               亚洲 d 日收盘时还不知道，用于亚洲资产时应取 lag=1
    """
    s = pd.Series(signal).astype(float).sort_index()
    looked = s.reindex(pd.DatetimeIndex(target_index) - pd.Timedelta(days=lag), method="ffill")
    return pd.Series(looked.values, index=target_index).fillna(0).astype(bool)


def us_equity_signal(vol_index, real_yield, vol_window=YEAR, ry_window=2 * YEAR, q=0.99):
    """标普500 / 纳指100：恐慌指数 或 实际利率。"""
    a = exceeds(vol_index, vol_window, q) & rising(vol_index)
    b = exceeds(real_yield, ry_window, q) & rising(real_yield)
    idx = a.index.union(b.index)
    return a.reindex(idx, fill_value=False) | b.reindex(idx, fill_value=False)


def gold_signal(real_yield, window=63, q=0.99):
    return exceeds(real_yield, window, q) & rising(real_yield)


def oil_signal(ovx, window=YEAR, q=0.99):
    return exceeds(ovx, window, q) & rising(ovx)


def hk_signal(usdcnh, real_yield, fx_window=2 * YEAR, ry_window=YEAR, q=0.99):
    a = exceeds(usdcnh, fx_window, q)
    b = exceeds(real_yield, ry_window, q) & rising(real_yield)
    idx = a.index.intersection(b.index)
    return a.reindex(idx) & b.reindex(idx)


def a_share_signal(div_yield, bond_yield, index_close, pca, value_window=4 * YEAR, q=0.10,
                   short=21, long=63):
    """沪深300 / 中证500：股债性价比低 且 指数近 5 日下跌 且 PcaScore 短均 > 长均。"""
    spread = (pd.Series(div_yield) - pd.Series(bond_yield)).dropna()
    cheap_lost = below(spread, value_window, q)
    falling = pd.Series(index_close).pct_change(5) < 0
    herding = pca.rolling(short).mean() > pca.rolling(long).mean()
    idx = cheap_lost.index.intersection(falling.index).intersection(herding.index)
    return cheap_lost.reindex(idx) & falling.reindex(idx) & herding.reindex(idx)


HORIZONS = (1, 2, 3, 5, 10)


def forward_returns(price, horizons=HORIZONS):
    """t 日收盘到 t+h 日收盘的收益。"""
    p = pd.Series(price).dropna()
    return pd.DataFrame({h: p.shift(-h) / p - 1 for h in horizons})

In [26]:
def event_study(signal, price, horizons=HORIZONS, period=None):
    """信号日与非信号日的未来 h 日平均收益（原文口径：逐日统计，重叠的信号日都算）。"""
    fwd = forward_returns(price, horizons)
    sig = align(signal, fwd.index)
    if period is not None:
        fwd, sig = fwd.loc[period], sig.loc[period]
    return pd.DataFrame({"预警时期": fwd[sig].mean(), "正常时期": fwd[~sig].mean()}).T


def count_episodes(signal, gap=10):
    """把信号日合并成独立事件：相邻信号日间隔不超过 gap 个交易日视为同一事件。"""
    days = np.flatnonzero(np.asarray(signal, bool))
    if len(days) == 0:
        return 0
    return int(1 + (np.diff(days) > gap).sum())


def episode_labels(signal, gap=10):
    """给每个信号日标上所属独立事件的编号（与 count_episodes 的归并规则相同），非信号日为 NaN。"""
    sig = pd.Series(signal).astype(bool)
    pos = np.flatnonzero(sig.values)
    lab = np.full(len(sig), np.nan)
    if len(pos):
        lab[pos] = np.concatenate([[0], np.cumsum(np.diff(pos) > gap)])
    return pd.Series(lab, sig.index)


def drop_worst_episode(signal, price, horizon=5, period=None, gap=10):
    """
    去掉对信号日平均收益「贡献最负」的那一次独立事件后，重算信号日的未来 horizon 日平均收益。
    用来检验结论是否只靠一次极端行情撑着。返回 (全部, 去掉后, 被去掉事件的起止日期)。
    """
    fwd = forward_returns(price, (horizon,))[horizon]
    sig = align(signal, fwd.index)
    if period is not None:
        fwd, sig = fwd.loc[period], sig.loc[period]
    lab = episode_labels(sig, gap)
    contrib = fwd[sig].groupby(lab[sig]).sum()
    worst = contrib.idxmin()
    keep = sig & (lab != worst)
    days = lab.index[lab == worst]
    return float(fwd[sig].mean()), float(fwd[keep].mean()), (days.min(), days.max())


def timing_returns(signal, returns, hold=5):
    """信号日之后 hold 个交易日空仓（收益记 0），其余满仓。"""
    r = pd.Series(returns).dropna()
    sig = align(signal, r.index).astype(int)
    flat = sig.shift(1, fill_value=0).rolling(hold, min_periods=1).max().astype(bool)
    return r.where(~flat, 0.0), flat


def placebo_event_study(signal, price, horizon=5, n=1000, seed=0, period=None, min_shift=YEAR):
    """
    安慰剂：把信号序列整体循环平移（保留信号的天数和扎堆结构，只打乱时点），
    得到「同样频率、随机时点」的信号日未来 horizon 日平均收益分布。
    返回 (真实值, 随机分布)。
    """
    fwd = forward_returns(price, (horizon,))[horizon]
    sig = align(signal, fwd.index)
    if period is not None:
        fwd, sig = fwd.loc[period], sig.loc[period]
    valid = fwd.notna().values
    f, s = fwd.values[valid], sig.values[valid]
    actual = f[s].mean()
    rng = np.random.default_rng(seed)
    dist = np.array([f[np.roll(s, rng.integers(min_shift, len(s) - min_shift))].mean() for _ in range(n)])
    return float(actual), dist

## 11. 因子拥挤度（Lee 2025）

```text
因子拥挤度与 alpha 衰减 —— Lee（2025）《Not All Factors Crowd Equally》（arXiv 2512.11913）。

══ 原文 v1（2025-12）的做法 ══════════════════════════════════════

    alpha 代理：滚动 36 个月夏普比率
    衰减模型：双曲线 α(t) = K/(1+λt)，与指数 K·e^(−λt)、线性 K − bt 对比 R²
    拥挤信号：在扩展窗口（至少 120 个月）上拟合双曲线，残差 = 实际夏普 − 预测夏普，负残差 = 拥挤
    尾部检验：1980–2000 年定阈值（残差中位数为拥挤分界，因子收益 10% 分位为崩盘线），
              2001–2024 年样本外比较拥挤 / 不拥挤时的崩盘概率

v1 未说明：时间 t 的起点、拟合方法、崩盘看当月还是下月收益、p 值所用的检验。

══ 时间错位（作者在 v3 中自己指出）══════════════════════════════

月份 t 的滚动夏普包含 t 月收益。t 月大跌 → 滚动夏普下降 → 残差变负 → 被判为「拥挤」。
若崩盘也看 t 月收益，「拥挤时崩盘更多」就只是同一个事件被数了两次，不是预测。
本模块对崩盘同时提供当月（horizon=0）与下月（horizon=1）两种口径，并在纯随机数据上走同一流程作对照。
```

In [27]:
DECAY = {
    "双曲线": (lambda t, K, lam: K / (1 + lam * t), [0.5, 0.1], ([-np.inf, 0], [np.inf, np.inf])),
    "指数": (lambda t, K, lam: K * np.exp(-lam * t), [0.5, 0.05], ([-np.inf, 0], [np.inf, np.inf])),
    "线性": (lambda t, K, b: K - b * t, [0.5, 0.01], ([-np.inf, -np.inf], [np.inf, np.inf])),
}


def rolling_sharpe(r, window=36):
    """年化滚动夏普：均值 / 标准差 × √12。t 月的值包含 t 月收益。"""
    r = pd.Series(r).dropna()
    return (r.rolling(window).mean() / r.rolling(window).std() * np.sqrt(12)).dropna()


def years_since(index, origin):
    return np.asarray((pd.DatetimeIndex(index) - pd.Timestamp(origin)).days / 365.25, float)


def fit_decay(t, y, kind="双曲线"):
    """非线性最小二乘拟合衰减曲线，返回 (参数, R²)。拟合失败返回 (None, nan)。"""
    f, p0, bounds = DECAY[kind]
    t, y = np.asarray(t, float), np.asarray(y, float)
    try:
        p, _ = curve_fit(f, t, y, p0=p0, bounds=bounds, maxfev=20000)
    except (RuntimeError, ValueError):
        return None, np.nan
    resid = y - f(t, *p)
    return p, float(1 - resid.var() / y.var()) if y.var() > 0 else np.nan


def decay_r2(sharpe, start="1995-01-01", end="2024-12-31", origin="1995-01-01", positive_only=True):
    """v1 表 2：在 [start, end] 的滚动夏普上分别拟合三种衰减曲线，返回 R²。"""
    s = sharpe.loc[start:end]
    if positive_only:
        s = s[s > 0]
    t = years_since(s.index, origin)
    return {k: fit_decay(t, s.values, k)[1] for k in DECAY}


def crowding_residual(sharpe, min_obs=120, kind="双曲线"):
    """
    v1 的拥挤信号：在 t 月，用严格早于 t 的全部滚动夏普（至少 min_obs 个）拟合衰减曲线，
    预测 t 月的值；残差 = 实际 − 预测（负 = 拥挤）。时间以第一个完整窗口为零点（年）。
    """
    s = sharpe.dropna()
    t_all = years_since(s.index, s.index[0])
    f = DECAY[kind][0]
    out = {}
    for i in range(min_obs, len(s)):
        p, _ = fit_decay(t_all[:i], s.values[:i], kind)
        if p is not None:
            out[s.index[i]] = s.values[i] - f(t_all[i], *p)
    return pd.Series(out)


def crash_table(returns, signal, horizon=0, train=("1980-01-01", "2000-12-31"), test=("2001-01-01", "2024-12-31"),
                crowded_below=True):
    """
    v1 表 4 的结构：
        拥挤 = signal 低于训练期中位数（crowded_below=False 时为高于）
        崩盘 = t+horizon 月收益低于训练期收益的 10% 分位
    在测试期统计 P(崩盘|拥挤)、P(崩盘|不拥挤)、比值，以及 Fisher 精确检验（双侧）p 值。
    """
    r = pd.Series(returns).dropna()
    sig = pd.Series(signal).dropna()
    med = sig.loc[train[0]:train[1]].median()
    thr = r.loc[train[0]:train[1]].quantile(0.10)
    crash = (r < thr).astype(int).shift(-horizon)
    crowded = (sig < med) if crowded_below else (sig > med)
    df = pd.concat({"拥挤": crowded, "崩盘": crash}, axis=1).loc[test[0]:test[1]].dropna()
    is_c = df["拥挤"].astype(bool)                               # 拼接后为 object 类型，取反前先转成布尔
    c, u = df.loc[is_c, "崩盘"], df.loc[~is_c, "崩盘"]
    table = [[int(c.sum()), int(len(c) - c.sum())], [int(u.sum()), int(len(u) - u.sum())]]
    p_c, p_u = c.mean(), u.mean()
    return {"P(崩盘|拥挤)": p_c, "P(崩盘|不拥挤)": p_u, "比值": p_c / p_u if p_u > 0 else np.nan,
            "Fisher p": fisher_exact(table)[1], "拥挤月数": len(c), "不拥挤月数": len(u)}

In [28]:
def null_crash_ratios(mu, sigma, n_months, n_sims=200, window=36, min_obs=120, seed=0, horizons=(0, 1),
                      train_frac=(16 / 61, 37 / 61), test_frac=(38 / 61, 61 / 61)):
    """
    纯随机对照：收益独立同分布 N(mu, sigma²)，没有任何拥挤或衰减。
    走与真实数据完全相同的流程（滚动夏普 → 扩展窗口双曲线残差 → 训练期定阈值 → 测试期统计），
    返回每次模拟、每个 horizon 的「拥挤 / 不拥挤」崩盘概率比值。
    训练、测试期按真实数据的相对位置切分。
    """
    rng = np.random.default_rng(seed)
    idx = pd.date_range("1963-07-31", periods=n_months, freq="ME")
    tr = (idx[int(train_frac[0] * n_months)], idx[int(train_frac[1] * n_months) - 1])
    te = (idx[int(test_frac[0] * n_months)], idx[int(test_frac[1] * n_months) - 1])
    rows = []
    for _ in range(n_sims):
        r = pd.Series(rng.normal(mu, sigma, n_months), idx)
        res = crowding_residual(rolling_sharpe(r, window), min_obs)
        rows.append({h: crash_table(r, res, h, train=tr, test=te)["比值"] for h in horizons})
    return pd.DataFrame(rows)

## 自检

原 pytest 回归测试，改写成断言。每格开头的 `%%selftest` 只在直接运行本 notebook 时执行；
被研究 notebook `%run` 引入时跳过。任何一条失败都会抛出 AssertionError。

In [29]:
from IPython.core.magic import register_cell_magic

_SELFTEST_LOG = []


class approx:
    """pytest.approx 的最小替代：只给 rel 或 abs 时只用那一个容差，都不给时 rel=1e-6、abs=1e-12。"""

    def __init__(self, expected, rel=None, abs=None):
        self.expected, self.rel, self.abs = expected, rel, abs

    def __eq__(self, actual):
        rel = self.rel if self.rel is not None else (1e-6 if self.abs is None else 0.0)
        tol_abs = self.abs if self.abs is not None else (1e-12 if self.rel is None else 0.0)
        e, a = np.asarray(self.expected, float), np.asarray(actual, float)
        return bool(np.all(np.abs(a - e) <= np.maximum(rel * np.abs(e), tol_abs)))

    def __repr__(self):
        return f"approx({self.expected!r}, rel={self.rel}, abs={self.abs})"


def parametrize(names, values):
    """pytest.mark.parametrize 的最小替代：记下参数组合，由 %%selftest 逐组调用。"""
    keys = [k.strip() for k in names.split(",")]

    def deco(f):
        f._cases = [dict(zip(keys, v if isinstance(v, tuple) else (v,))) for v in values]
        return f
    return deco


@register_cell_magic
def selftest(line, cell):
    if not RUN_SELFTEST:
        return
    ip = get_ipython()
    before = set(ip.user_ns)
    ip.run_cell(cell).raise_error()
    for name in [k for k in ip.user_ns if k not in before and k.startswith("test_")]:
        f = ip.user_ns[name]
        for kw in getattr(f, "_cases", [{}]):
            f(**kw)
        _SELFTEST_LOG.append(name)
        print("✓", name)

In [30]:
%%selftest
# copula
def _grid(n=800):
    g = np.linspace(0.5 / n, 1 - 0.5 / n, n)
    return np.meshgrid(g, g)


@parametrize("C, theta", [
    (ClaytonCopula, 0.01), (ClaytonCopula, 2.0),
    (GumbelCopula, 1.0), (GumbelCopula, 1.8),
])
def test_density_integrates_to_one(C, theta):
    U, V = _grid()
    assert np.exp(C.loglik_terms(theta, U, V)).mean() == approx(1.0, abs=5e-3)


@parametrize("C, theta0", [(ClaytonCopula, 1e-9), (GumbelCopula, 1.0)])
def test_independence_loglik_is_zero(C, theta0):
    U, V = _grid(50)
    assert np.abs(C.loglik_terms(theta0, U, V)).max() < 1e-6


def test_clayton_recovers_theta():
    """条件分布法抽 Clayton(θ=2) 样本，MLE 应估回 θ≈2，λ_D≈2^-0.5。"""
    rng = np.random.default_rng(0)
    n, th = 4000, 2.0
    u, w = rng.uniform(size=n), rng.uniform(size=n)
    v = (u ** (-th) * (w ** (-th / (1 + th)) - 1) + 1) ** (-1 / th)
    c = ClaytonCopula().fit(u, v)
    assert c.theta == approx(th, rel=0.1)
    assert c.lambda_raw == approx(2 ** -0.5, abs=0.03)


def test_no_overflow_at_extreme_ranks():
    """θ 大、u 极小时 u^-θ 会溢出；log 空间实现必须给出有限似然。"""
    u = np.array([1e-6, 0.5, 1 - 1e-6])
    assert np.isfinite(ClaytonCopula.loglik_terms(30.0, u, u)).all()
    assert np.isfinite(GumbelCopula.loglik_terms(30.0, u, u)).all()


def test_negative_dependence_gives_zero_lambda():
    """负相关窗口：两族都应落在独立边界上，λ=0（两种判零口径都是）。"""
    rng = np.random.default_rng(1)
    z1 = rng.standard_normal(60)
    z2 = -0.6 * z1 + 0.8 * rng.standard_normal(60)
    res = fit_window(z1, z2)
    for rule in ZERO_RULES:
        assert zero_out(res["lambda_D_raw"], res["p_D"], rule) == 0
        assert zero_out(res["lambda_U_raw"], res["p_U"], rule) == 0


def test_lrtest_false_positive_rate_under_independence():
    """LR 口径下，独立数据的 λ_D>0 占比不应超过名义 5%（χ²(1) 偏保守）。"""
    b = noise_baseline(window=60, n_rep=600, rho=0.0, seed=7, zero_rule="lrtest")
    assert b["lambda_D>0占比"] <= 0.07


def test_positive_correlation_inflates_lambda_without_tail_dependence():
    """
    已知局限的「回归测试」：高斯 copula（无尾部相关）ρ=0.5 时，
    λ_D>0 仍几乎总是成立。若这条失败，说明判零逻辑变了，结论需要重审。
    """
    b = noise_baseline(window=60, n_rep=300, rho=0.5, seed=8, zero_rule="lrtest")
    assert b["lambda_D>0占比"] > 0.85


def test_rank_pit_open_interval():
    x = np.array([3.0, 1.0, 2.0, 2.0])
    u = rank_pit(x)
    assert (u > 0).all() and (u < 1).all()
    assert u[2] == u[3]                      # ties 取平均秩


def test_rolling_window_has_no_lookahead():
    """改动窗口之后的数据，不应影响此前任何一天的估计。"""
    import pandas as pd

    rng = np.random.default_rng(2)
    idx = pd.bdate_range("2020-01-01", periods=150)
    x, y = pd.Series(rng.standard_normal(150), idx), pd.Series(rng.standard_normal(150), idx)
    r1 = rolling_tail_dependence(x, y, window=60)
    y2 = y.copy()
    y2.iloc[120:] *= -5
    r2 = rolling_tail_dependence(x, y2, window=60)
    cut = idx[119]
    pd.testing.assert_frame_equal(r1.loc[:cut], r2.loc[:cut])

✓ test_density_integrates_to_one
✓ test_independence_loglik_is_zero
✓ test_clayton_recovers_theta
✓ test_no_overflow_at_extreme_ranks
✓ test_negative_dependence_gives_zero_lambda


✓ test_lrtest_false_positive_rate_under_independence


✓ test_positive_correlation_inflates_lambda_without_tail_dependence
✓ test_rank_pit_open_interval


✓ test_rolling_window_has_no_lookahead


In [31]:
%%selftest
# elliptical
@parametrize("rho, nu", [(0.5, 4.0), (-0.3, 10.0)])
def test_t_density_integrates_to_one(rho, nu):
    """copula 密度 = 二元 t 密度 / 两个一元 t 密度；漏除一次归一化常数时积分约 0.38。"""
    n = 400
    g = np.linspace(0.5 / n, 1 - 0.5 / n, n)
    U, V = np.meshgrid(g, g)
    dens = np.array([np.exp(t_loglik(rho, nu, np.array([a]), np.array([b])))
                     for a, b in zip(U.ravel()[::23], V.ravel()[::23])])
    assert dens.mean() == approx(1.0, abs=0.05)


def test_t_recovers_parameters():
    rng = np.random.default_rng(0)
    n, rho, nu = 3000, 0.4, 4.0
    z = rng.multivariate_normal([0, 0], [[1, rho], [rho, 1]], n)
    x = z / np.sqrt(rng.chisquare(nu, n) / nu)[:, None]
    r = t_vs_gaussian(rank_pit(x[:, 0]), rank_pit(x[:, 1]))
    assert r["rho_t"] == approx(rho, abs=0.05)
    assert 2.5 < r["nu"] < 6.5
    assert r["p_t"] < 1e-6


def test_t_lr_size_under_gaussian():
    """高斯数据下，5% 水平的拒绝率应接近名义水平（边界上用 ½χ²(1) 校准）。"""
    s = gaussian_null_t_test(window=250, n_sims=300, rho=0.2, seed=5)
    assert 0.01 <= (s.p_t < 0.05).mean() <= 0.10


def test_t_tail_dependence_formula():
    assert t_tail_dependence(0.0, NU_MAX) == 0.0
    # ρ=0.5, ν=4：λ = 2·t_5(−√(5·0.5/1.5)) ≈ 0.2532
    assert t_tail_dependence(0.5, 4.0) == approx(0.2532, abs=1e-3)

✓ test_t_density_integrates_to_one


✓ test_t_recovers_parameters


✓ test_t_lr_size_under_gaussian
✓ test_t_tail_dependence_formula


In [32]:
%%selftest
# sjc
def _jc_cdf(u, v, tu, tl):
    k, g = 1 / np.log2(2 - tu), -1 / np.log2(tl)
    A = (1 - (1 - u) ** k) ** (-g) + (1 - (1 - v) ** k) ** (-g) - 1
    return 1 - (1 - A ** (-1 / g)) ** (1 / k)


@parametrize("tu, tl", [(0.3, 0.6), (0.05, 0.2), (0.7, 0.1)])
def test_jc_density_matches_cdf_mixed_derivative(tu, tl):
    e = 1e-4
    for u, v in [(0.2, 0.3), (0.9, 0.85), (0.05, 0.6)]:
        fd = (_jc_cdf(u + e, v + e, tu, tl) - _jc_cdf(u + e, v - e, tu, tl)
              - _jc_cdf(u - e, v + e, tu, tl) + _jc_cdf(u - e, v - e, tu, tl)) / (4 * e * e)
        assert np.exp(jc_logpdf(np.array(u), np.array(v), tu, tl)) == approx(fd, rel=1e-3)


@parametrize("tu, tl", [(0.3, 0.6), (0.7, 0.1), (1e-4, 1e-4)])
def test_sjc_density_integrates_to_one(tu, tl):
    n = 500
    g = np.linspace(0.5 / n, 1 - 0.5 / n, n)
    U, V = np.meshgrid(g, g)
    assert np.exp(sjc_logpdf(U, V, tu, tl)).mean() == approx(1.0, abs=0.01)


def test_finite_in_upper_corner_with_large_tau():
    """κ 大时 (1−u)^κ 会下溢，曾导致 w=0、似然为 +inf，优化器被带偏。"""
    u = np.array([0.999, 0.9999, 0.5])
    assert np.isfinite(sjc_logpdf(u, u, TAU_MAX, TAU_MAX)).all()


def test_recovers_pure_lower_tail():
    """Clayton(θ=1)：λ_L = 0.5，λ_U = 0；不对称检验应强烈拒绝。"""
    rng = np.random.default_rng(0)
    n = 3000
    u, w = rng.uniform(size=n), rng.uniform(size=n)
    v = (u ** -1.0 * (w ** -0.5 - 1) + 1) ** -1.0
    r = asymmetry_test(rank_pit(u), rank_pit(v))
    assert r["tau_L"] == approx(0.5, abs=0.08)
    assert r["tau_U"] < 0.05
    assert r["p_asym"] < 1e-10


def test_asymmetry_size_under_gaussian():
    """对称的高斯数据（ρ=0.3）下，5% 水平误报率应接近名义水平。"""
    rng = np.random.default_rng(1)
    p = []
    for _ in range(200):
        z1 = rng.standard_normal(250)
        z2 = 0.3 * z1 + np.sqrt(1 - 0.09) * rng.standard_normal(250)
        p.append(asymmetry_test(rank_pit(z1), rank_pit(z2))["p_asym"])
    assert 0.01 <= np.nanmean(np.array(p) < 0.05) <= 0.10

✓ test_jc_density_matches_cdf_mixed_derivative


✓ test_sjc_density_integrates_to_one
✓ test_finite_in_upper_corner_with_large_tau


✓ test_recovers_pure_lower_tail


✓ test_asymmetry_size_under_gaussian


In [33]:
%%selftest
# signals_backtest
def test_to_returns_does_not_fill_gaps():
    px = pd.DataFrame({"a": [1.0, 2.0, np.nan, 4.0], "b": [1.0, 1.1, 1.2, 1.3]})
    r = to_returns(px)["a"]
    assert np.isnan(r.loc[2]) and np.isnan(r.loc[3])     # 缺失日及其后一日都不应有收益


def test_state_machine_trigger_and_release():
    idx = pd.bdate_range("2021-01-01", periods=6)
    ld = pd.Series([0, .2, .2, 0, 0, 0], idx, dtype=float)
    lu = pd.Series([0, 0, .1, 0, 0, 0], idx, dtype=float)
    dif_a = pd.Series([1, -1, -1, -1, -1, -1], idx, dtype=float)
    dif_b = pd.Series([1, 1, 1, -1, 1, 1], idx, dtype=float)
    s = warning_state(ld, lu, dif_a, dif_b)
    # 第 2 天触发；第 3 天 λ_U>0 但已在预警中（只看解除条件）；
    # 第 4 天 λ_D=0 但债 DIF<0 不解除；第 5 天解除
    assert s["预警中"].tolist() == [False, True, True, True, False, False]


def test_episodes_and_matching():
    idx = pd.bdate_range("2013-06-17", periods=10)
    state = pd.Series([0, 0, 0, 1, 1, 1, 0, 0, 0, 0], idx).astype(bool)
    ep = episodes(state)
    assert len(ep) == 1 and ep.loc[0, "持续交易日"] == 3
    m, extra = match_episodes(ep)
    assert m.loc[m["触发"] == "2013-06-20", "触发偏差(天)"].item() == 0
    assert extra.empty
    assert episodes(pd.Series(False, idx)).empty


def test_overlay_uses_signal_next_day():
    idx = pd.bdate_range("2021-01-01", periods=4)
    base = pd.Series([0.01, 0.01, 0.01, 0.01], idx)
    warn = pd.Series([False, True, False, False], idx)
    out = overlay_warning(base, warn, cash_rate=0.0)
    assert out.tolist() == [0.01, 0.01, 0.0, 0.01]        # t 日预警 → t+1 日空仓


def test_to_daily_forward_fills_sparse_signal():
    daily = pd.bdate_range("2021-01-01", periods=10)
    sparse = pd.Series([True, False], index=daily[[0, 5]])
    d = to_daily(sparse, daily)
    assert d.iloc[:5].all() and not d.iloc[5:].any()


def test_risk_parity_weights_are_lagged_and_inverse_vol():
    rng = np.random.default_rng(0)
    idx = pd.bdate_range("2020-01-01", periods=300)
    r = pd.DataFrame({"e": rng.normal(0, .02, 300), "b": rng.normal(0, .002, 300)}, idx)
    w = risk_parity_weights(r, lookback=60, target_vol=None)
    month_end = r.groupby(r.index.to_period("M")).tail(1).index[3]
    nxt = r.index[r.index.get_loc(month_end) + 1]
    vol = r.loc[:month_end].tail(60).std()
    expect = (1 / vol) / (1 / vol).sum()
    np.testing.assert_allclose(w.loc[nxt].values, expect.values, rtol=1e-6)
    assert w.loc[month_end].ne(w.loc[nxt]).any()            # 调仓日当天仍用旧权重


def test_pairwise_cash_overlay_rules():
    idx = pd.bdate_range("2021-01-01", periods=4)
    r = pd.DataFrame({"股": 0.01, "债": 0.002, "金": 0.005}, index=idx)
    w = {"股": 0.2, "债": 0.6, "金": 0.2}
    warn = {("股", "债"): pd.Series([False, True, True, False], idx),
            ("股", "金"): pd.Series([False, False, True, False], idx)}
    out, n = pairwise_cash_overlay(r, w, warn, cash_rate=0.0)
    # t 日状态 → t+1 日执行：第 2 天 n=0；第 3 天 n=1（股债转现金，只剩金 0.2）；第 4 天 n=2 全现金
    assert n.tolist() == [0, 0, 1, 2]
    np.testing.assert_allclose(out.values, [0.0042, 0.0042, 0.2 * 0.005, 0.0])

✓ test_to_returns_does_not_fill_gaps
✓ test_state_machine_trigger_and_release
✓ test_episodes_and_matching
✓ test_overlay_uses_signal_next_day
✓ test_to_daily_forward_fills_sparse_signal


✓ test_risk_parity_weights_are_lagged_and_inverse_vol
✓ test_pairwise_cash_overlay_rules


In [34]:
%%selftest
# allocation
from scipy.stats import t as student_t


def _returns(n=600, k=4, seed=0):
    rng = np.random.default_rng(seed)
    C = 0.5 * np.ones((k, k)) + 0.5 * np.eye(k)
    X = rng.multivariate_normal(np.zeros(k), C, n) * np.linspace(0.005, 0.02, k)
    return pd.DataFrame(X, columns=[f"a{i}" for i in range(k)])


def test_two_asset_formula_limits():
    # d=0（完全尾部相依）：无分散效果，等于线性加权
    assert two_asset_tr(0.5, 0.02, 0.04, 0.0) == approx(0.03)
    # 单一资产：w1=1 时就是该资产自身的 TR
    assert two_asset_tr(1.0, 0.02, 0.04, 5.0) == approx(0.02)
    # 距离越大，合成风险越小
    assert two_asset_tr(0.5, 0.03, 0.03, 9.0) < two_asset_tr(0.5, 0.03, 0.03, 0.5)


def test_tree_tr_reduces_to_formula_for_two_assets():
    Z = np.array([[0, 1, 1.0, 2]])
    D = np.array([[0.0, 2.0], [2.0, 0.0]])
    tr = np.array([0.02, 0.05])
    assert tree_tr(np.array([0.3, 0.7]), tr, D, Z) == approx(two_asset_tr(0.3, 0.02, 0.05, 2.0))


def test_min_cvar_beats_alternatives():
    R = _returns()
    w = min_cvar(R, q=0.95, upper=1.0)
    assert w.sum() == approx(1.0) and (w >= -1e-9).all()
    es_opt = es_hist(R @ w)
    rng = np.random.default_rng(1)
    for _ in range(200):
        v = rng.dirichlet(np.ones(R.shape[1]))
        assert es_hist(R @ v) >= es_opt - 1e-4


def test_weight_functions_respect_bounds():
    R = _returns(k=6)
    for f in (min_variance, min_cvar):
        w = f(R, upper=0.25)
        assert w.sum() == approx(1.0) and w.max() <= 0.25 + 1e-6
    h = hrp_weights(R)
    assert h.sum() == approx(1.0) and (h > 0).all()


def test_es_gpd_matches_student_t():
    """t(4) 分布 99% ES 的理论值；GPD 外推应与之接近（大样本）。"""
    rng = np.random.default_rng(3)
    r = student_t.rvs(4, size=200_000, random_state=rng)
    q, nu = 0.99, 4
    x = student_t.ppf(q, nu)
    es_true = student_t.pdf(x, nu) / (1 - q) * (nu + x ** 2) / (nu - 1)
    assert es_gpd(r, q=q, threshold=0.95) == approx(es_true, rel=0.05)
    assert es_hist(r, q=q) == approx(es_true, rel=0.05)


def test_cap_weights():
    w = cap_weights(pd.Series([0.5, 0.3, 0.1, 0.1]), upper=0.3)
    assert w.sum() == approx(1.0) and w.max() <= 0.3 + 1e-9
    assert w.iloc[2] == approx(w.iloc[3])            # 超额按原比例分配


def test_erc_equalizes_vol_contributions():
    R = _returns(k=5)
    w = erc_weights(R)
    rc = vol_contributions(w, R)
    np.testing.assert_allclose(rc / rc.sum(), 0.2, atol=1e-3)


def test_es_parity_equalizes_es_contributions_with_hedge_asset():
    """含一个与股票负相关的「避险」资产：其 ES 贡献在等权时为负，TRP 解仍应使各贡献相等且为正。"""
    rng = np.random.default_rng(7)
    T = 3000
    eq = rng.standard_t(4, (T, 2)) * 0.012 + rng.standard_t(4, (T, 1)) * 0.008
    hedge = -0.3 * eq[:, :1] + rng.normal(0, 0.003, (T, 1))
    X = pd.DataFrame(np.hstack([eq, hedge]), columns=["股1", "股2", "避险"])
    assert es_contributions(np.full(3, 1 / 3), X)[2] < 0
    w = es_parity_weights(X, q=0.95)
    rc = es_contributions(w.values, X, q=0.95)
    assert (rc > 0).all()
    np.testing.assert_allclose(rc / rc.sum(), 1 / 3, atol=0.03)


def test_es_parity_close_to_erc_for_gaussian():
    """椭圆分布下 ES 贡献与波动率贡献成比例，两种风险平价的权重应几乎相同。"""
    R = _returns(n=20000, k=4, seed=8)
    np.testing.assert_allclose(es_parity_weights(R).values, erc_weights(R).values, atol=0.02)

✓ test_two_asset_formula_limits
✓ test_tree_tr_reduces_to_formula_for_two_assets


✓ test_min_cvar_beats_alternatives


✓ test_weight_functions_respect_bounds


✓ test_es_gpd_matches_student_t
✓ test_cap_weights


✓ test_erc_equalizes_vol_contributions


✓ test_es_parity_equalizes_es_contributions_with_hedge_asset


✓ test_es_parity_close_to_erc_for_gaussian


In [35]:
%%selftest
# simulation
from scipy.stats import kurtosis, ks_2samp


def _skewed_hist(T=2000, seed=0):
    rng = np.random.default_rng(seed)
    z = rng.multivariate_normal([0, 0, 0], [[1, .6, .3], [.6, 1, .4], [.3, .4, 1]], T)
    H = np.exp(0.6 * z)
    return (H - H.mean(0)) / H.std(0) * np.array([.01, .02, .015])


def test_gtja_recursion_inflates_later_variances():
    """整理稿记载的递推式：第 1 位资产不受影响，后面与之相关的资产方差被高估。"""
    H = _skewed_hist()
    X = gtja_recursion(H, 100_000, np.random.default_rng(1))
    ratio = np.diag(np.cov(X, rowvar=False)) / np.diag(np.cov(H, rowvar=False))
    assert ratio[0] < 1.10
    assert ratio[1] > 1.20 and ratio[2] > 1.10


@parametrize("method", [gaussian_copula_kde, iman_conover])
def test_correct_methods_preserve_marginals_and_order_free(method):
    H = _skewed_hist()
    rng = np.random.default_rng(2)
    X = method(H, 100_000, rng)
    ratio = np.diag(np.cov(X, rowvar=False)) / np.diag(np.cov(H, rowvar=False))
    assert np.all(np.abs(ratio - 1) < 0.08)                       # 仅 KDE 带宽带来的少量平滑
    for j in range(3):
        assert ks_2samp(X[:, j], H[:, j]).statistic < 0.03
        # 样本峰度在厚尾下抽样波动很大，KDE 平滑也会略降峰度，容差取 25%；
        # 作为对照，递推式会把第 2 位资产的峰度稀释约 40%
        assert kurtosis(X[:, j]) == approx(kurtosis(H[:, j]), rel=0.25)
    # 顺序无关：倒序排列后，各资产的方差误差不变
    Xr = method(H[:, ::-1], 100_000, rng)[:, ::-1]
    ratio_r = np.diag(np.cov(Xr, rowvar=False)) / np.diag(np.cov(H, rowvar=False))
    np.testing.assert_allclose(ratio, ratio_r, atol=0.05)


def test_kde_ppf_inverts_cdf():
    x = np.random.default_rng(3).standard_normal(500)
    q = kde_ppf(x, np.array([0.05, 0.5, 0.95]))
    assert q[0] < q[1] < q[2]
    assert q[1] == approx(np.median(x), abs=0.1)


def test_simulate_horizon_shape():
    H = _skewed_hist(300)
    P = simulate_horizon(iman_conover, H, horizon=20, n_paths=50, rng=np.random.default_rng(4))
    assert P.shape == (50, 20, 3)


def test_simplex_grid():
    g = simplex_grid(3, 0.25)
    assert len(g) == 15                                      # C(4+2, 2)
    np.testing.assert_allclose(g.sum(axis=1), 1.0)
    assert (g >= 0).all()


def test_path_risk_decomposition_and_values():
    rng = np.random.default_rng(5)
    paths = rng.normal(0.0005, 0.01, (2000, 20, 2))
    W = simplex_grid(2, 0.1)
    r = path_risk(paths, W)
    np.testing.assert_allclose(r["TR"], r["IH"] + r["EH"])     # 总风险 = 期内 + 期末（互斥）
    # 对单一资产直接按对数路径计算，应与向量化结果一致
    logP = np.log(np.cumprod(1 + paths[:, :, 0], axis=1))
    ih = np.minimum(logP.min(1), 0) <= -0.01
    eh = ~ih & (logP[:, -1] <= 0.01)
    i = int(np.flatnonzero((W[:, 0] == 1.0))[0])
    assert r["IH"][i] == approx(ih.mean()) and r["EH"][i] == approx(eh.mean())


def test_optimizer_picks_obvious_asset():
    """一个资产稳定上涨（几乎不会触发风险），另一个高波动：应全部配给前者。"""
    rng = np.random.default_rng(6)
    steady = np.full((1500, 20, 1), 0.001) + rng.normal(0, 0.0002, (1500, 20, 1))
    wild = rng.normal(0, 0.03, (1500, 20, 1))
    w, risk = gtja_optimal_weights(np.concatenate([steady, wild], axis=2), simplex_grid(2, 0.05))
    assert w[0] == approx(1.0) and risk["TR"] < 0.05

✓ test_gtja_recursion_inflates_later_variances


✓ test_correct_methods_preserve_marginals_and_order_free


✓ test_kde_ppf_inverts_cdf
✓ test_simulate_horizon_shape
✓ test_simplex_grid
✓ test_path_risk_decomposition_and_values
✓ test_optimizer_picks_obvious_asset


In [36]:
%%selftest
# riskmodels
from scipy.stats import norm
from scipy.stats import t as student_t


def test_normal_var_es_known_values():
    r = np.random.default_rng(0).standard_normal(400_000)
    v, _ = var_es_normal(r, 0.99)
    _, e = var_es_normal(r, 0.975)
    assert v == approx(norm.ppf(0.99), abs=0.01)
    assert e == approx(norm.pdf(norm.ppf(0.975)) / 0.025, abs=0.01)     # ≈ 2.338


def test_t_and_gpd_match_student_t_tail():
    r = student_t.rvs(4, size=200_000, random_state=np.random.default_rng(1))
    q = 0.99
    x = student_t.ppf(q, 4)
    es_true = student_t.pdf(x, 4) / (1 - q) * (4 + x ** 2) / 3
    assert var_es_t(r, q)[1] == approx(es_true, rel=0.05)
    assert var_es_gpd(r, q)[1] == approx(es_true, rel=0.05)


def test_kupiec_and_christoffersen():
    rng = np.random.default_rng(2)
    good = rng.random(5000) < 0.01
    assert kupiec(good, 0.01)[1] > 0.05
    assert kupiec(rng.random(5000) < 0.03, 0.01)[1] < 1e-6
    clustered = np.zeros(5000, bool)
    for s in range(100, 5000, 500):
        clustered[s:s + 5] = True                                      # 突破成串出现
    assert christoffersen(clustered)[1] < 1e-6


def test_z2_zero_when_correct_negative_when_es_too_low():
    rng = np.random.default_rng(3)
    x = rng.standard_normal(200_000)
    a = 0.025
    v = np.full_like(x, norm.ppf(1 - a))
    e = np.full_like(x, norm.pdf(norm.ppf(1 - a)) / a)
    assert abs(acerbi_szekely_z2(x, v, e, a)) < 0.03
    assert acerbi_szekely_z2(x, v, e * 0.7, a) < -0.3


def test_gjr_filter_constant_when_no_dynamics():
    p = {"mu": 0.0, "omega": 4.0, "alpha": 0.0, "gamma": 0.0, "beta": 0.0}
    sd, nxt = gjr_filter(np.random.default_rng(4).normal(0, 0.02, 50), p)
    assert np.allclose(sd[1:], 0.02) and nxt == approx(0.02)   # √4 % = 2%


def test_rolling_forecasts_no_lookahead_and_shape():
    rng = np.random.default_rng(5)
    idx = pd.bdate_range("2015-01-01", periods=420)
    r = pd.Series(rng.standard_t(5, 420) * 0.01, idx)
    f1 = rolling_forecasts(r, window=300, refit=40)
    r2 = r.copy(); r2.iloc[400:] *= 5
    f2 = rolling_forecasts(r2, window=300, refit=40)
    cut = idx[399]
    pd.testing.assert_frame_equal(f1.loc[:cut].drop(columns=("收益", "")), f2.loc[:cut].drop(columns=("收益", "")))
    assert len(f1) == 120 and (f1.xs("ES", axis=1, level=1) > 0).all().all()

✓ test_normal_var_es_known_values


✓ test_t_and_gpd_match_student_t_tail
✓ test_kupiec_and_christoffersen
✓ test_z2_zero_when_correct_negative_when_es_too_low
✓ test_gjr_filter_constant_when_no_dynamics


✓ test_rolling_forecasts_no_lookahead_and_shape


In [37]:
%%selftest
# warning
def test_exceeds_uses_only_past_data():
    idx = pd.bdate_range("2020-01-01", periods=300)
    x = pd.Series(np.arange(300, dtype=float), idx)          # 单调上升：每天都创新高
    e = exceeds(x, 100, 0.99)
    assert not e.iloc[:100].any()                           # 窗口未满不出信号
    assert e.iloc[100:].all()
    x2 = x.copy(); x2.iloc[200] = -1                         # 当日骤降不应触发
    assert not exceeds(x2, 100, 0.99).iloc[200]


def test_rising_slope():
    x = pd.Series([1, 2, 3, 4, 5, 4, 3, 2, 1, 0], dtype=float)
    r = rising(x, 5)
    assert r.iloc[4] and not r.iloc[9]


def test_pca_score_bounds_and_herding():
    rng = np.random.default_rng(0)
    idx = pd.bdate_range("2020-01-01", periods=200)
    common = rng.standard_normal((200, 1))
    indep = pd.DataFrame(rng.standard_normal((200, 16)), idx)
    herd = pd.DataFrame(common * 3 + rng.standard_normal((200, 16)) * 0.3, idx)
    a, b = pca_score(indep).dropna(), pca_score(herd).dropna()
    assert ((a > 0) & (a <= 1)).all() and ((b > 0) & (b <= 1)).all()
    assert b.mean() > 0.8 > a.mean()                        # 同向走势时前两大主成分占比高


def test_timing_flat_after_signal():
    idx = pd.bdate_range("2020-01-01", periods=12)
    r = pd.Series(0.01, idx)
    sig = pd.Series(False, idx); sig.iloc[2] = True
    out, flat = timing_returns(sig, r, hold=5)
    assert flat.tolist() == [False] * 3 + [True] * 5 + [False] * 4   # 信号次日起空仓 5 天
    assert (out[flat] == 0).all()


def test_align_lag_and_ffill():
    us = pd.Series([False, True, False], pd.to_datetime(["2020-01-02", "2020-01-03", "2020-01-06"]))
    asia = pd.to_datetime(["2020-01-03", "2020-01-06", "2020-01-07"])
    assert align(us, asia).tolist() == [True, False, False]
    assert align(us, asia, lag=1).tolist() == [False, True, False]   # 美国 1/3 的信息亚洲 1/6 才能用


def test_episodes():
    s = np.zeros(100, bool); s[[5, 6, 8, 40, 41, 90]] = True
    assert count_episodes(s, gap=10) == 3


def test_episode_labels_and_drop_worst():
    idx = pd.bdate_range("2020-01-01", periods=100)
    sig = pd.Series(False, idx); sig.iloc[[5, 6, 40, 41, 80]] = True
    lab = episode_labels(sig)
    assert lab.dropna().tolist() == [0, 0, 1, 1, 2]
    price = pd.Series(100.0, idx); price.iloc[46:] = 50.0        # 第二次事件之后暴跌
    allm, kept, (a, b) = drop_worst_episode(sig, price, horizon=5)
    assert allm < 0 and kept == approx(0.0) and a == idx[40] and b == idx[41]

✓ test_exceeds_uses_only_past_data
✓ test_rising_slope
✓ test_pca_score_bounds_and_herding
✓ test_timing_flat_after_signal
✓ test_align_lag_and_ffill
✓ test_episodes
✓ test_episode_labels_and_drop_worst


In [38]:
%%selftest
# crowding
def test_rolling_sharpe_includes_current_month():
    idx = pd.date_range("2000-01-31", periods=40, freq="ME")
    r = pd.Series(0.01, idx); r.iloc[-1] = -0.20
    s = rolling_sharpe(r + np.random.default_rng(0).normal(0, 1e-3, 40), 36)
    assert s.iloc[-1] < s.iloc[-2]                          # 当月大跌立刻拉低当月的滚动夏普


def test_fit_decay_recovers_hyperbola():
    t = np.linspace(0, 30, 300)
    y = 0.8 / (1 + 0.2 * t) + np.random.default_rng(1).normal(0, 0.01, 300)
    p, r2 = fit_decay(t, y, "双曲线")
    assert p[0] == approx(0.8, rel=0.05) and p[1] == approx(0.2, rel=0.1) and r2 > 0.95


def test_residual_uses_only_past_fits():
    rng = np.random.default_rng(2)
    idx = pd.date_range("1960-01-31", periods=300, freq="ME")
    s = pd.Series(rng.normal(0.5, 0.2, 300), idx)
    a = crowding_residual(s, min_obs=120)
    s2 = s.copy(); s2.iloc[250:] += 5
    b = crowding_residual(s2, min_obs=120)
    pd.testing.assert_series_equal(a.iloc[:130], b.iloc[:130])   # 第 250 个月之后的变化不影响此前的残差


def test_crash_table_counts():
    idx = pd.date_range("1980-01-31", periods=48, freq="ME")
    year = [0.01, 0.02, 0.0, 0.015, 0.005, 0.01, 0.03, -0.01, 0.0, 0.02, 0.01, -0.08]   # 每年只有 12 月大跌
    r = pd.Series(np.tile(year, 4), idx)
    sig = pd.Series(np.where(r < -0.05, -1.0, 1.0), idx)       # 每次大跌当月信号都低
    t = crash_table(r, sig, 0, train=("1980", "1981"), test=("1982", "1983"))
    assert t["P(崩盘|拥挤)"] == 1.0 and t["P(崩盘|不拥挤)"] == 0.0
    t1 = crash_table(r, sig, 1, train=("1980", "1981"), test=("1982", "1983"))
    assert t1["P(崩盘|拥挤)"] == 0.0                            # 下月口径：大跌之后的 1 月不是崩盘

✓ test_rolling_sharpe_includes_current_month


✓ test_fit_decay_recovers_hyperbola


✓ test_residual_uses_only_past_fits
✓ test_crash_table_counts


In [39]:
if RUN_SELFTEST:
    print(f"自检通过：{len(_SELFTEST_LOG)} 项")

自检通过：58 项
